# 🌙 Low-Light Image Enhancement bằng CNN nhẹ — DarkIR-Edge (C=24)
**Môn học:** Xử Lý Ảnh và Thị Giác Máy Tính (CS2203) — Master of Science in Computer Science 2026 (UIT - ĐHQG TP.HCM)  
**Phiên bản:** `v4` — mở rộng đa miền dữ liệu huấn luyện (`6.777 Train pool` [639 LOLv2-Real + 2.380 LSRW-Huawei + 1.858 LOL-Blur + 1.000 LSRW-Nikon + 900 LOLv2-Synthetic] / `150 Val` [50:50:50 giữ nguyên v3] / `190 Test` [giữ nguyên 100% v3]), rút mẫu theo **quota cố định `4.800` mẫu/epoch** (`1.278 lolv2` + `1.122 huawei` + `1.600 blur` + `400 nikon` + `400 lolv2syn` = `600 steps/epoch`), **tắt Early Stopping** (`use_early_stopping=False`) và huấn luyện cố định **70 epoch** (`42.000 steps`, `T_max=42.000`) cho **DarkIR-Edge** và **Standard U-Net** (2 seed `[42, 123]`; bỏ `no-FFT`). Bổ sung kiểm định ghép cặp đối chứng `v3 ↔ v4` (`paired_tests_v3_v4.csv`) và đánh giá không tham chiếu trên `Real-LOLBlur` (482 ảnh). Thư mục Drive: `MyDrive/llie_project_v4/`.  
**Môi trường:** Google Colab (GPU NVIDIA Tesla T4 / L4)

---

### 🧭 Cấu trúc Notebook (Tách bạch Mô hình Đề xuất và Mô hình Đối chuẩn)
1. **Cell 1 – 6 (Setup & Dữ liệu):** Khởi tạo, giải nén `llie_data_clean_v4.zip` vào ổ SSD cục bộ `/content/datasets`, nạp `split_manifest.json` cố định (`6.777 Train pool / 4.800 mẫu mỗi epoch` + `150 Val` + `190 Test`) và bộ nạp dữ liệu xác định (`MASTER_SEED = 42`).
2. **Cell 8 (Kiến trúc đề xuất `DarkIR-Edge`):** Đặt riêng biệt trong một cell độc lập, chú thích tường minh từng khối của **DarkIR (CVPR 2025)** (`FreMLP`, `Di-SpAM`, `SCA`, `SimpleGate`, `Side Supervision`) cùng **2 cải tiến vệ tinh**:
   - **Cải tiến vệ tinh 1 (Lấy cảm hứng từ HVI-CIDNet CVPR 2025 & Lý thuyết Retinex):** Trích xuất kênh độ rọi cực đại **Max-RGB Illumination Prior ($I_{\max} = \max(R,G,B)$)** ghép vào đầu vào tạo thành tensor 4 kênh (`Intro Conv 4 -> C`), kết hợp hàm mất mát **Cosine Color Loss ($\mathcal{L}_{\text{color}}$)** ở Cell 11 để triệt tiêu lệch màu sRGB mà không làm tăng tham số suy luận.
   - **Cải tiến vệ tinh 2 (Kế thừa từ Edge-LLIE CVPRW 2024):** Cổng lọc nhiễu dư **Residual Gated Skip Connection** (`GatedSkip`) trên 3 tầng nối tắt để chặn nhiễu hạt cảm biến smartphone tràn sang Decoder.
3. **Cell 9 (Baseline Đối chuẩn Tùy chọn — `Standard U-Net`):** Tách riêng sang cell độc lập để chỉ dùng khi cần đối chuẩn.
4. **Cell 14 (Huấn luyện `DarkIR-Edge`):** Thực thi huấn luyện cho kiến trúc đề xuất.
5. **Cell 15 (Tùy chọn — Huấn luyện `Standard U-Net`):** Tách riêng hoàn toàn, điều khiển bằng cờ `RUN_UNET_BASELINE = True/False`.
6. **Cell 16 – 25 (Đánh giá định lượng, Kiểm định thống kê, Biểu đồ & Hiệu năng phần cứng).**

In [1]:
# ===== CELL 1: Imports, Hạt giống Cố định (MASTER_SEED) & Cấu hình Toàn cục =====
import os, sys, re, json, glob, time, math, random, hashlib, copy, zipfile, shutil, subprocess
from pathlib import Path
import numpy as np, cv2, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler, ConcatDataset
try:
    import psutil
except ImportError:
    psutil = None

IN_COLAB = 'google.colab' in sys.modules
TEST_MODE = os.environ.get('LLIE_TEST') == '1'          # Chỉ bật khi chạy kiểm thử tự động ngoài Colab
BASE = '/content' if IN_COLAB else os.environ.get('LLIE_BASE', './llie_run')
DRIVE = '/content/drive/MyDrive/llie_project_v4' if IN_COLAB else f'{BASE}/drive'

# -------------------------------------------------------------------------
# QUẢN LÝ HẠT GIỐNG NGẪU NHIÊN CỐ ĐỊNH (DETERMINISTIC SEEDING)
# - MASTER_SEED = 42 được khóa cứng ngay từ đầu để mọi thao tác chia dữ liệu,
#   khởi tạo trọng số và cắt patch (Random Crop) ở các lần chạy lại cùng seed
#   đều tái lập chính xác 100% (không có bất kỳ sự ngẫu nhiên trôi dạt nào).
# -------------------------------------------------------------------------
MASTER_SEED = 42

def set_seed(seed=MASTER_SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(MASTER_SEED)

DEV = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = DEV.type == 'cuda'

# -------------------------------------------------------------------------
# CẤU HÌNH PHIÊN BẢN THỰC NGHIỆM & QUẢN LÝ VERSION TRÊN GOOGLE DRIVE
# - Cho phép train đi train lại nhiều lần với các siêu tham số, loss weights
#   hoặc thử nghiệm khác nhau mà KHÔNG bị ghi đè hay mất mát dữ liệu cũ.
# - Mỗi version lưu riêng biệt tại:
#     /content/drive/MyDrive/llie_project_v4/experiments/{EXP_VERSION}/
#       ├── run_config.json    (siêu tham số, cấu hình chi tiết của version)
#       ├── checkpoints/       (chứa best.pt, last.pt, final.pt, history.csv)
#       └── results/           (chứa csv, biểu đồ png, ảnh phục chế mẫu)
# - Đồng thời tự động cập nhật bản tóm tắt mới nhất vào:
#     /content/drive/MyDrive/llie_project_v4/results_latest/
# -------------------------------------------------------------------------
EXP_VERSION = 'v4'             # Đổi tên khi chạy lần mới: 'v1', 'v2', 'v3', 'v4', v.v.
FORCE_RETRAIN = False          # Đặt True để huấn luyện lại từ đầu trong cùng version (bỏ qua final.pt)

EXP_DIR    = f'{DRIVE}/experiments/{EXP_VERSION}'
CKPT_DIR   = f'{EXP_DIR}/checkpoints'
RESULT_DIR = f'{EXP_DIR}/results'
LATEST_DIR = f'{DRIVE}/results_latest'

CFG = dict(
    version    = EXP_VERSION,
    force_retrain = FORCE_RETRAIN,
    zip_path   = f'{DRIVE}/llie_data_clean_v4.zip',       # File zip dữ liệu trên Google Drive
    data_root  = f'{BASE}/datasets',                   # Thư mục giải nén trên ổ SSD cục bộ của Colab
    exp_dir    = EXP_DIR,                              # Thư mục riêng của version trên Google Drive
    ckpt_dir   = CKPT_DIR,                             # Thư mục lưu checkpoint theo version
    result_dir = RESULT_DIR,                           # Thư mục lưu kết quả csv/png theo version
    latest_dir = LATEST_DIR,                           # Thư mục đồng bộ kết quả mới nhất trên Drive
    manifest   = f'{DRIVE}/split_manifest.json',       # File phân chia dữ liệu dùng chung bất biến
    # --- Siêu tham số huấn luyện cố định v4 (khớp ke_hoach_v4.md & ke_hoach_v4_notebook.md) ---
    epochs=70, batch_size=8, patch=256,                # 4,800 mẫu/epoch / 8 = 600 steps/epoch -> 42,000 steps cố định (70 epoch, T_max=42,000)
    quota=dict(lolv2=1278, huawei=1122, blur=1600, nikon=400, lolv2syn=400),  # Tổng = 4,800 mẫu/epoch (bảo toàn 1,278 lolv2 + 1,122 huawei như v3)
    lr=2e-4, lr_min=1e-6, weight_decay=1e-4, betas=(0.9, 0.999),
    val_interval=5, ema_decay=0.999, grad_clip=1.0, num_workers=4,
    loss_w=dict(side=0.5, ssim=0.2, color=0.1),
    # --- Phân bổ pool dữ liệu v4 theo manifest (Train pool=6,777 : Val=150 : Test=190) ---
    n_train=dict(lolv2=639, huawei=2380, blur=1858, nikon=1000, lolv2syn=900),
    n_val=dict(lolv2=50, huawei=50, blur=50),
    huawei_gap=10,
    v3_results_dir=f'{DRIVE}/v3_reference_results',    # Nơi đặt results_per_image.csv & results_extra.csv của v3 để chạy đối chứng v3 <-> v4
    realblur_zip=f'{DRIVE}/real_lolblur_eval.zip',     # Gói 482 ảnh Real-LOLBlur độc lập (không có GT)
    # --- Danh sách hạt giống đầy đủ của thử nghiệm v4 (dùng cho các cell đánh giá; chọn seed chạy từng phiên trực tiếp ở Cell 14 & Cell 15) ---
    # • Phiên 1: đặt SEEDS_TO_RUN_DARKIR_EDGE = [42] (Cell 14) và SEEDS_TO_RUN_UNET = [42] (Cell 15)
    # • Phiên 2: đặt SEEDS_TO_RUN_DARKIR_EDGE = [123] (Cell 14) và SEEDS_TO_RUN_UNET = [123] (Cell 15); các cell đánh giá tự quét cả [42, 123]
    darkir_edge_seeds=[42, 123],                       # DarkIR-Edge (seed 42, 123)
    darkir_edge_nofft_seeds=[],                        # v4: Bỏ chạy no-FFT
    unet_seeds=[42, 123],                              # Standard U-Net đối chuẩn (seed 42, 123)
    unet_base=32,                                      # base=32 -> 1,928,483 params (khớp 1,924,038 của DarkIR-Edge C=24)
    # --- Quy tắc dừng v3 (đăng ký trước): TẮT Early Stopping, chạy cố định 70 epoch cho mọi mô hình ---
    # Vẫn đo val PSNR macro (EMA, 50:50:50 = 150 ảnh Val) mỗi 5 epoch để lưu last.pt (resume) và best.pt (phân tích phụ).
    use_early_stopping=False,
    early_stop_patience=15,
    early_stop_min_delta=0.05,
    early_stop_min_epochs=40,
    max_steps_per_epoch=None,
)

if TEST_MODE:
    CFG.update(
        epochs=2, batch_size=2, patch=64, val_interval=1, num_workers=0,
        quota=dict(lolv2=6, huawei=6, blur=8, nikon=2, lolv2syn=2),
        n_train=dict(lolv2=3, huawei=6, blur=6, nikon=4, lolv2syn=4),
        n_val=dict(lolv2=2, huawei=2, blur=2), huawei_gap=1,
        darkir_edge_seeds=[42, 123], darkir_edge_nofft_seeds=[], unet_seeds=[42, 123], max_steps_per_epoch=4,
        early_stop_patience=1, early_stop_min_epochs=1
    )

os.makedirs(CFG['data_root'], exist_ok=True)
if not IN_COLAB:
    for d in (DRIVE, CFG['exp_dir'], CFG['ckpt_dir'], CFG['result_dir'], CFG['latest_dir']):
        os.makedirs(d, exist_ok=True)

try:
    import lpips
    LPIPS_OK = True
except Exception:
    if IN_COLAB:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'lpips'])
    try:
        import lpips; LPIPS_OK = True
    except Exception as e:
        LPIPS_OK = False; print('WARN: Không có thư viện lpips -> tự động bỏ qua chỉ số LPIPS.', e)

print(f'PyTorch {torch.__version__} | Device: {DEV} | AMP: {USE_AMP} | MASTER_SEED: {MASTER_SEED}')
print(f'Phiên bản thực nghiệm (Version): "{CFG["version"]}"')
print(f'Thư mục đích Google Drive: {CFG["exp_dir"]}')

PyTorch 2.11.0+cu130 | Device: cuda | AMP: True | MASTER_SEED: 42
Phiên bản thực nghiệm (Version): "v4"
Thư mục đích Google Drive: /content/drive/MyDrive/llie_project_v4/experiments/v4


In [2]:
# ===== CELL 2: Kết nối Google Drive & Giải nén dữ liệu vào ổ SSD cục bộ (/content/datasets) =====
if IN_COLAB:
    # Nếu trước đó có thư mục cục bộ /content/drive chưa mount gây xung đột thì dọn sạch
    if os.path.exists('/content/drive') and not os.path.ismount('/content/drive'):
        shutil.rmtree('/content/drive')
    from google.colab import drive
    drive.mount('/content/drive')
    # Khởi tạo đầy đủ cấu trúc thư mục trên Google Drive (bao gồm phân tầng version)
    for d in (DRIVE, CFG['exp_dir'], CFG['ckpt_dir'], CFG['result_dir'], CFG['latest_dir']):
        os.makedirs(d, exist_ok=True)
    # Lưu bản ghi cấu hình run_config.json vào thư mục của version này trên Drive để phục vụ truy vết khoa học
    cfg_record = {k: (str(v) if isinstance(v, (torch.device, Path)) else v) for k, v in CFG.items()}
    with open(f"{CFG['exp_dir']}/run_config.json", 'w', encoding='utf-8') as f_cfg:
        json.dump(cfg_record, f_cfg, indent=2, ensure_ascii=False)
    print(f'✔ Đã mount Google Drive! Thư mục lưu kết quả phiên bản "{CFG["version"]}":')
    print(f'   -> Checkpoints: {CFG["ckpt_dir"]}')
    print(f'   -> Results:     {CFG["result_dir"]}')

    if not os.listdir(CFG['data_root']):
        assert os.path.exists(CFG['zip_path']), f"Không tìm thấy file zip tại: {CFG['zip_path']}"
        print('Đang copy và giải nén', CFG['zip_path'], 'vào SSD cục bộ...')
        shutil.copy(CFG['zip_path'], '/content/llie_data.zip')
        with zipfile.ZipFile('/content/llie_data.zip') as z:
            z.extractall(CFG['data_root'])
        # Nếu trong zip đã đóng gói sẵn split_manifest.json chuẩn từ máy local thì copy sang DRIVE
        bundled_man = Path(CFG['data_root']) / 'split_manifest.json'
        if bundled_man.exists() and not os.path.exists(CFG['manifest']):
            shutil.copy(str(bundled_man), CFG['manifest'])
            print('Đã nạp split_manifest.json đóng gói sẵn từ file zip vào Google Drive.')
    print('Dữ liệu sẵn sàng tại:', CFG['data_root'], os.listdir(CFG['data_root']))
elif TEST_MODE:
    fake_specs = {
        'lolv2_real_train': 6, 'lolv2_real_test': 4,
        'lsrw_huawei_train': 10, 'lsrw_huawei_test': 3,
        'lsrw_nikon_train': 4, 'lsrw_nikon_test': 3,
        'lolv2_syn_train': 4,
        'lolblur_train': 8, 'lolblur_test': 3,
    }
    for k, n in fake_specs.items():
        ld = Path(CFG['data_root']) / 'fake' / k / 'low'
        hd = Path(CFG['data_root']) / 'fake' / k / 'high'
        ld.mkdir(parents=True, exist_ok=True); hd.mkdir(parents=True, exist_ok=True)
        for i in range(1, n + 1):
            H, W = (70, 92) if i % 2 == 0 else (68, 84)
            hi_img = np.random.randint(80, 240, (H, W, 3), dtype=np.uint8)
            lo_img = np.clip((hi_img.astype(np.float32) / 255.0) ** 2.5 * 255.0 + np.random.randn(H, W, 3) * 5, 0, 255).astype(np.uint8)
            cv2.imwrite(str(ld / f'low{i:05d}.png'), lo_img)
            cv2.imwrite(str(hd / f'normal{i:05d}.png'), hi_img)
    print('Đã khởi tạo dữ liệu giả lập cho TEST_MODE.')
else:
    print('Chạy cục bộ ngoài Colab. Thư mục dữ liệu:', CFG['data_root'])

Mounted at /content/drive
✔ Đã mount Google Drive! Thư mục lưu kết quả phiên bản "v4":
   -> Checkpoints: /content/drive/MyDrive/llie_project_v4/experiments/v4/checkpoints
   -> Results:     /content/drive/MyDrive/llie_project_v4/experiments/v4/results
Đang copy và giải nén /content/drive/MyDrive/llie_project_v4/llie_data_clean_v4.zip vào SSD cục bộ...
Dữ liệu sẵn sàng tại: /content/datasets ['DICM', 'split_manifest.json', 'LOL-v2', 'LOLBlur', 'LIME', 'LSRW']


In [3]:
# ===== CELL 3: Kiểm tra cấu trúc cây thư mục thực tế sau khi giải nén =====
def show_tree(root, depth=4, max_items=5):
    root = Path(root)
    if not root.exists():
        print('Thư mục chưa tồn tại:', root); return
    def rec(p, d):
        if d > depth: return
        kids = sorted(p.iterdir())
        dirs = [k for k in kids if k.is_dir()]
        files = [k for k in kids if k.is_file()]
        for k in dirs:
            print('  ' * d + f'[D] {k.name}/')
            rec(k, d + 1)
        if files:
            sample = ', '.join(f.name for f in files[:max_items])
            print('  ' * d + f'    ({len(files)} files) vd: {sample}')
    rec(root, 0)

show_tree(CFG['data_root'])

[D] DICM/
      (69 files) vd: 01.JPG, 02.JPG, 03.JPG, 04.JPG, 05.JPG
[D] LIME/
      (10 files) vd: 1.bmp, 10.bmp, 2.bmp, 3.bmp, 4.bmp
[D] LOL-v2/
  [D] Real_captured/
    [D] Test/
      [D] Low/
            (100 files) vd: low00690.png, low00691.png, low00692.png, low00693.png, low00694.png
      [D] Normal/
            (100 files) vd: normal00690.png, normal00691.png, normal00692.png, normal00693.png, normal00694.png
    [D] Train/
      [D] Low/
            (689 files) vd: low00001.png, low00002.png, low00003.png, low00004.png, low00005.png
      [D] Normal/
            (689 files) vd: normal00001.png, normal00002.png, normal00003.png, normal00004.png, normal00005.png
  [D] Synthetic/
    [D] Train/
      [D] Low/
            (900 files) vd: r000da54ft.png, r001d260dt.png, r002fc3e2t.png, r00444b95t.png, r005f3e70t.png
      [D] Normal/
            (900 files) vd: r000da54ft.png, r001d260dt.png, r002fc3e2t.png, r00444b95t.png, r005f3e70t.png
[D] LOLBlur/
  [D] test/
    [D] high_s

### 🔒 Phân chia Dữ liệu Cố định (`MASTER_SEED = 42`) & Bảo đảm Zero Data Leakage
- Trong Cell 5 dưới đây:
  - Bộ **LOLv2-Real** (`689` cặp Train): Lấy cố định **50 cặp cuối** làm Validation, **639 cặp đầu** làm Train (tránh rò rỉ các ảnh chụp liên tiếp cùng một căn phòng).
  - Bộ **LSRW-Huawei** (`1.000` cặp Train): Lấy cố định **50 cặp cuối** làm Validation, chừa vùng đệm `gap = 10` ảnh, và dùng đối tượng `random.Random(MASTER_SEED)` với `MASTER_SEED = 42` để rút cố định **561 cặp** làm Train.
  - **Mở rộng đa miền v4 (`llie_data_clean_v4.zip`):** Giữ nguyên 100% tập `Val` (150 cặp) và `Test` (190 cặp) của v3; mở rộng `Train` lên 5 miền: `639` LOLv2-Real + `2.380` LSRW-Huawei (cố định Val `00951..01000`, loại đệm `00941..00950` và `01001..01010`) + `1.858` LOL-Blur (trên đúng 150 sequence Train của v3 với `gap >= 4`, bao hàm 400 cặp v3) + `1.000` LSRW-Nikon (chọn đều stride $\approx 3.13$ từ 3.150 cặp, không trùng tên và SHA-256 với 20 ảnh Eval Nikon; ghi nhận `nikon_test_in_domain=True`) + `900` LOLv2-Synthetic ($384\times 384$).
- Mọi đường dẫn trong `split_manifest.json` được lưu ở dạng **đường dẫn tương đối** so với `CFG['data_root']` kèm mã băm **SHA-256**, đảm bảo tạo ở máy local hay trên Google Colab đều ra **cùng một danh sách ảnh giống hệt nhau 100%**.

In [4]:
# ===== CELL 5: Ghép cặp ảnh + Chia Train/Val/Test cố định (MASTER_SEED=42) + Lưu split_manifest.json =====
CANDIDATE_PATHS = {
    'lolv2_real_train': [
        ('LOL-v2/Real_captured/Train/Low', 'LOL-v2/Real_captured/Train/Normal'),
        ('Real_captured/Train/Low',        'Real_captured/Train/Normal'),
        ('LOLv2-Real/Train/Low',           'LOLv2-Real/Train/Normal'),
    ],
    'lolv2_real_test': [
        ('LOL-v2/Real_captured/Test/Low',  'LOL-v2/Real_captured/Test/Normal'),
        ('LOL-v2/Real_captured/test/Low',  'LOL-v2/Real_captured/test/Normal'),
        ('Real_captured/Test/Low',         'Real_captured/Test/Normal'),
        ('LOLv2-Real/Test/Low',            'LOLv2-Real/Test/Normal'),
    ],
    'lsrw_huawei_train': [
        ('LSRW/Train/Huawei/low',          'LSRW/Train/Huawei/high'),
        ('LSRW/Training data/Huawei/low',  'LSRW/Training data/Huawei/high'),
        ('data1/Training data/Huawei/low', 'data1/Training data/Huawei/high'),
        ('Train/Huawei/low',               'Train/Huawei/high'),
        ('LSRW/Huawei/Train/low',          'LSRW/Huawei/Train/high'),
    ],
    'lsrw_huawei_test': [
        ('LSRW/Eval/Huawei/low',           'LSRW/Eval/Huawei/high'),
        ('data1/Eval/Huawei/low',          'data1/Eval/Huawei/high'),
        ('LSRW_/Eval/Huawei/low',          'LSRW_/Eval/Huawei/high'),
        ('Eval/Huawei/low',                'Eval/Huawei/high'),
        ('LSRW/Huawei/Test/low',           'LSRW/Huawei/Test/high'),
    ],
    'lolblur_train':   [('LOLBlur/train/low_blur_noise', 'LOLBlur/train/high_sharp_scaled')],   # chỉ dùng cho TEST_MODE (dữ liệu thật đọc từ manifest)
    'lolblur_test':    [('LOLBlur/test/low_blur_noise',  'LOLBlur/test/high_sharp_scaled')],
    'lsrw_nikon_train': [('LSRW/Train/Nikon/low',        'LSRW/Train/Nikon/high')],
    'lolv2_syn_train':  [('LOL-v2/Synthetic/Train/Low',  'LOL-v2/Synthetic/Train/Normal')],
    'lsrw_nikon_test': [
        ('LSRW/Eval/Nikon/low',            'LSRW/Eval/Nikon/high'),
        ('data1/Eval/Nikon/low',           'data1/Eval/Nikon/high'),
        ('LSRW_/Eval/Nikon/low',           'LSRW_/Eval/Nikon/high'),
        ('Eval/Nikon/low',                 'Eval/Nikon/high'),
        ('LSRW/Nikon/Test/low',            'LSRW/Nikon/Test/high'),
    ],
}

def resolve_sources(data_root):
    if TEST_MODE:
        return {k: (f'fake/{k}/low', f'fake/{k}/high') for k in CANDIDATE_PATHS}
    resolved = {}
    root = Path(data_root)
    for key, cands in CANDIDATE_PATHS.items():
        found = None
        for lo_rel, hi_rel in cands:
            if (root / lo_rel).is_dir() and (root / hi_rel).is_dir():
                found = (lo_rel, hi_rel); break
        assert found is not None, f"Không tìm thấy thư mục cho '{key}' trong {root}. Kiểm tra lại cấu trúc thư mục ở Cell 3."
        resolved[key] = found
    return resolved

IMG_EXT = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}
def _nat(s): return [int(t) if t.isdigit() else t for t in re.split(r'(\d+)', s)]
def _key(stem): return re.sub(r'(?i)^(low|normal|high|gt)[_\-. ]*|[_\-. ]*(low|normal|high|gt)$', '', stem).strip('_-. ')

def make_pairs(low_rel, high_rel):
    root = Path(CFG['data_root'])
    ld, hd = root / low_rel, root / high_rel
    assert ld.is_dir() and hd.is_dir(), f'Không thấy thư mục: {ld} hoặc {hd}'
    low  = {_key(p.stem): p.relative_to(root).as_posix() for p in ld.iterdir() if p.suffix.lower() in IMG_EXT}
    high = {_key(p.stem): p.relative_to(root).as_posix() for p in hd.iterdir() if p.suffix.lower() in IMG_EXT}
    diff = set(low) ^ set(high)
    assert not diff, f'{len(diff)} ảnh không khớp cặp giữa {ld} và {hd}, vd: {sorted(diff, key=_nat)[:5]}'
    return [(low[k], high[k]) for k in sorted(low, key=_nat)]

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''): h.update(chunk)
    return h.hexdigest()

def to_abs_pairs(rel_pairs):
    root = Path(CFG['data_root'])
    res = []
    for l, h in rel_pairs:
        lp = Path(l) if Path(l).is_absolute() and Path(l).exists() else (root / l)
        hp = Path(h) if Path(h).is_absolute() and Path(h).exists() else (root / h)
        res.append((str(lp), str(hp)))
    return res

def build_manifest():
    root = Path(CFG['data_root'])
    nv, nt, gap = CFG['n_val'], CFG['n_train'], CFG['huawei_gap']
    lol = make_pairs(*SOURCES['lolv2_real_train'])
    hw  = make_pairs(*SOURCES['lsrw_huawei_train'])

    # 1. LOLv2-Real: 50 cặp cuối làm Val, 639 cặp đầu làm Train
    lol_val, lol_pool = lol[-nv['lolv2']:], lol[:-nv['lolv2']]
    if len(lol_pool) <= nt['lolv2']:
        lol_train = lol_pool
    else:
        idx = sorted(random.Random(MASTER_SEED).sample(range(len(lol_pool)), nt['lolv2']))
        lol_train = [lol_pool[i] for i in idx]

    # 2. LSRW-Huawei: 50 cặp cuối làm Val, vùng đệm gap=10, lấy mẫu cố định bằng MASTER_SEED=42
    hw_val, hw_pool = hw[-nv['huawei']:], hw[:len(hw) - nv['huawei'] - gap]
    idx = sorted(random.Random(MASTER_SEED).sample(range(len(hw_pool)), nt['huawei']))
    hw_train = [hw_pool[i] for i in idx]

    # 3. LOL-Blur: chỉ dùng ở TEST_MODE (chia đơn giản trên dữ liệu giả); bản thật dùng manifest chọn theo sequence
    bl = make_pairs(*SOURCES['lolblur_train'])
    bl_val, bl_train = bl[-nv['blur']:], bl[:nt['blur']]

    nk_train = make_pairs(*SOURCES['lsrw_nikon_train'])[:nt['nikon']]
    syn_train = make_pairs(*SOURCES['lolv2_syn_train'])[:nt['lolv2syn']]
    splits_rel = {
        'train': {'lolv2': lol_train, 'huawei': hw_train, 'blur': bl_train, 'nikon': nk_train, 'lolv2syn': syn_train},
        'val':   {'lolv2': lol_val,   'huawei': hw_val,   'blur': bl_val},
        'test':  {
            'lolv2':  make_pairs(*SOURCES['lolv2_real_test']),
            'huawei': make_pairs(*SOURCES['lsrw_huawei_test']),
            'nikon':  make_pairs(*SOURCES['lsrw_nikon_test']),
            'blur':   make_pairs(*SOURCES['lolblur_test']),
        },
    }
    sets = {(s, d): {p[0] for p in v} for s, dd in splits_rel.items() for d, v in dd.items()}
    keys = list(sets)
    for i in range(len(keys)):
        for j in range(i + 1, len(keys)):
            assert not (sets[keys[i]] & sets[keys[j]]), f'PHÁT HIỆN RÒ RỈ DỮ LIỆU giữa {keys[i]} và {keys[j]}'

    man = {
        'nikon_test_in_domain': True,
        **{
            s: {d: [[l, h, sha256(root / l), sha256(root / h)] for l, h in v] for d, v in dd.items()}
            for s, dd in splits_rel.items()
        }
    }
    json.dump(man, open(CFG['manifest'], 'w'), indent=1)
    return {s: {d: to_abs_pairs(v) for d, v in dd.items()} for s, dd in splits_rel.items()}

if os.path.exists(CFG['manifest']) and not TEST_MODE:
    man = json.load(open(CFG['manifest']))
    assert man.get('nikon_test_in_domain') is True, 'Manifest v4 bắt buộc ghi chú nikon_test_in_domain=True!'
    SPLITS = {s: {d: to_abs_pairs([(x[0], x[1]) for x in v]) for d, v in man[s].items()} for s in ('train', 'val', 'test')}
    print('Đã nạp manifest v4 cố định có sẵn:', CFG['manifest'], '| nikon_test_in_domain =', man.get('nikon_test_in_domain'))
    # Chốt cấu hình v4 theo manifest thực tế
    EXPECTED_V4 = {
        'train': CFG['n_train'],
        'val':   {'lolv2': 50, 'huawei': 50, 'blur': 50},
        'test':  {'lolv2': 100, 'huawei': 30, 'nikon': 20, 'blur': 40},
    }
    got = {s: {d: len(v) for d, v in dd.items()} for s, dd in SPLITS.items()}
    assert got == EXPECTED_V4, f'Manifest không khớp cấu hình v4!\n  có:  {got}\n  cần: {EXPECTED_V4}'
    tr_blur_seqs = {Path(lp).parent.name for lp, _ in SPLITS['train']['blur']}
    va_blur_seqs = {Path(lp).parent.name for lp, _ in SPLITS['val']['blur']}
    assert not (tr_blur_seqs & va_blur_seqs), f'Rò rỉ sequence blur giữa train và val: {tr_blur_seqs & va_blur_seqs}'
else:
    assert TEST_MODE, f"Không thấy {CFG['manifest']}: manifest v4 phải nằm trong llie_data_clean_v4.zip (tạo bằng docs/ai/prepare_llie_data_v4.py)"
    SOURCES = resolve_sources(CFG['data_root'])
    SPLITS = build_manifest()
    print('TEST_MODE: đã tạo manifest v4 trên dữ liệu giả:', CFG['manifest'])

for s, dd in SPLITS.items():
    print(f'{s:5s}', {d: len(v) for d, v in dd.items()}, '-> Tổng:', sum(len(v) for v in dd.values()))

Đã nạp manifest v4 cố định có sẵn: /content/drive/MyDrive/llie_project_v4/split_manifest.json | nikon_test_in_domain = True
train {'lolv2': 639, 'huawei': 2380, 'blur': 1858, 'nikon': 1000, 'lolv2syn': 900} -> Tổng: 6777
val   {'lolv2': 50, 'huawei': 50, 'blur': 50} -> Tổng: 150
test  {'lolv2': 100, 'huawei': 30, 'nikon': 20, 'blur': 40} -> Tổng: 190


In [5]:
# ===== CELL 6: Dataset & DataLoader (Nạp lười miền lớn để bảo vệ RAM + QuotaSampler 4,800 mẫu/epoch) =====
def imread_rgb(p):
    im = cv2.imread(p, cv2.IMREAD_COLOR)
    if im is None: raise IOError(f'Không đọc được file ảnh: {p}')
    return cv2.cvtColor(im, cv2.COLOR_BGR2RGB)

def to_tensor(x):
    return torch.from_numpy(np.ascontiguousarray(x)).permute(2, 0, 1).float().div_(255.0)

class PairDataset(Dataset):
    """Hỗ trợ nạp RAM (lazy=False) cho các miền nhỏ và nạp lười từ đĩa (lazy=True) cho miền blur/huawei lớn."""
    def __init__(self, pairs, patch=256, train=True, lazy=False, domain='unknown'):
        self.pairs, self.patch, self.train, self.lazy, self.domain = list(pairs), patch, train, lazy, domain
        self.cache, nbytes = [], 0
        if not self.lazy:
            for lp, hp in self.pairs:
                lo, hi = imread_rgb(lp), imread_rgb(hp)
                assert lo.shape == hi.shape, f'Kích thước low/high không khớp: {lp} {lo.shape} vs {hi.shape}'
                self.cache.append((lo, hi))
                nbytes += lo.nbytes + hi.nbytes
        else:
            # Kiểm tra nhanh cặp đầu tiên để đảm bảo đường dẫn hợp lệ
            if self.pairs:
                lo0, hi0 = imread_rgb(self.pairs[0][0]), imread_rgb(self.pairs[0][1])
                assert lo0.shape == hi0.shape
        self.gb = nbytes / 1e9

    def __len__(self):
        return len(self.pairs)

    def _load_raw(self, idx):
        if not self.lazy:
            return self.cache[idx]
        lp, hp = self.pairs[idx]
        lo, hi = imread_rgb(lp), imread_rgb(hp)
        return lo, hi

    def __getitem__(self, i):
        lo, hi = self._load_raw(i % len(self.pairs))
        if self.train:
            H, W, _ = lo.shape; p = self.patch
            if H < p or W < p:
                ph, pw = max(0, p - H), max(0, p - W)
                lo = np.pad(lo, ((0, ph), (0, pw), (0, 0)), 'reflect')
                hi = np.pad(hi, ((0, ph), (0, pw), (0, 0)), 'reflect')
                H, W, _ = lo.shape
            y = int(torch.randint(0, H - p + 1, (1,)))
            x = int(torch.randint(0, W - p + 1, (1,)))
            lo, hi = lo[y:y + p, x:x + p], hi[y:y + p, x:x + p]
            if torch.rand(1) < 0.5: lo, hi = lo[:, ::-1], hi[:, ::-1]
            if torch.rand(1) < 0.5: lo, hi = lo[::-1], hi[::-1]
        return to_tensor(lo), to_tensor(hi)

class MultiDomainTrainDataset(Dataset):
    """Gộp các PairDataset theo từng miền; tự động bật lazy=True cho blur (và cho huawei khi > 1,000 cặp) theo §3 ke_hoach_v4_notebook.md."""
    def __init__(self, train_splits, patch=256):
        self.datasets = {}
        self.offsets = {}
        self.domain_ranges = {}
        cur = 0
        for dom in ('lolv2', 'huawei', 'blur', 'nikon', 'lolv2syn'):
            if dom not in train_splits:
                continue
            pairs = train_splits[dom]
            # Theo §3: nạp lười miền blur; nếu huawei > 1000 ảnh (2,380 ảnh = ~9.8 GB) thì cũng nạp lười để tổng RAM cache <= 6 GB
            use_lazy = (dom == 'blur') or (dom == 'huawei' and len(pairs) > 1000)
            ds = PairDataset(pairs, patch=patch, train=True, lazy=use_lazy, domain=dom)
            self.datasets[dom] = ds
            self.offsets[dom] = cur
            self.domain_ranges[dom] = (cur, cur + len(ds))
            cur += len(ds)
        self.total_pool = cur
        self.gb = sum(ds.gb for ds in self.datasets.values())

    def __len__(self):
        return self.total_pool

    def domain_of_global_idx(self, g_idx):
        for dom, (st, en) in self.domain_ranges.items():
            if st <= g_idx < en:
                return dom, g_idx - st
        raise IndexError(f'Global index out of range: {g_idx}')

    def __getitem__(self, g_idx):
        dom, local_idx = self.domain_of_global_idx(int(g_idx))
        return self.datasets[dom][local_idx]

class QuotaSampler(Sampler):
    """
    Rút mẫu theo quota cố định mỗi epoch (Mục 2 ke_hoach_v4_notebook.md):
    - Nối các hoán vị ngẫu nhiên (torch.randperm) của pool từng miền cho đến khi đủ quota (không lặp trong cùng 1 lượt).
    - Xáo trộn toàn bộ 4,800 chỉ số bằng torch.Generator(seed + epoch) để tái lập 100%.
    """
    def __init__(self, multi_ds: MultiDomainTrainDataset, quota: dict, seed: int = MASTER_SEED):
        self.multi_ds = multi_ds
        self.quota = dict(quota)
        self.seed = int(seed)
        self.epoch = 0
        self.total_samples = sum(self.quota.values())

    def set_epoch(self, epoch: int):
        self.epoch = int(epoch)

    def __len__(self):
        return self.total_samples

    def __iter__(self):
        g = torch.Generator()
        g.manual_seed(self.seed + self.epoch * 10007)
        all_indices = []
        for dom, q in self.quota.items():
            st, en = self.multi_ds.domain_ranges[dom]
            n_pool = en - st
            assert n_pool > 0, f'Pool của miền {dom} rỗng!'
            picked = []
            while len(picked) < q:
                perm = torch.randperm(n_pool, generator=g).tolist()
                need = q - len(picked)
                picked.extend(st + idx for idx in perm[:need])
            all_indices.extend(picked)
        assert len(all_indices) == self.total_samples
        global_perm = torch.randperm(len(all_indices), generator=g).tolist()
        return iter([all_indices[i] for i in global_perm])

def flat(split):
    return [p for d in SPLITS[split].values() for p in d]

train_ds = MultiDomainTrainDataset(SPLITS['train'], CFG['patch'])
val_ds   = PairDataset(flat('val'), train=False, lazy=False)
val_by_dom = {d: PairDataset(p, train=False, lazy=False, domain=d) for d, p in SPLITS['val'].items()}
ram_sys_gb = (psutil.virtual_memory().used / (1024 ** 3)) if psutil is not None else 0.0
print(f'train_ds pool: {len(train_ds)} cặp ({ {d: ( len(ds), "lazy" if ds.lazy else f"{ds.gb:.2f}GB" ) for d, ds in train_ds.datasets.items()} })')
print(f'Tổng RAM cache train_ds ≈ {train_ds.gb:.2f} GB | val_ds ≈ {val_ds.gb:.2f} GB | RAM hệ thống đang dùng ≈ {ram_sys_gb:.2f} GB')

def _seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

def make_train_loader(seed):
    g = torch.Generator()
    g.manual_seed(seed)
    sampler = QuotaSampler(train_ds, CFG['quota'], seed=seed)
    return DataLoader(
        train_ds, batch_size=CFG['batch_size'], sampler=sampler, drop_last=True,
        generator=g, worker_init_fn=_seed_worker,
        num_workers=CFG['num_workers'], pin_memory=DEV.type == 'cuda',
        persistent_workers=CFG['num_workers'] > 0
    )

steps_per_ep = sum(CFG['quota'].values()) // CFG['batch_size']
print(f'Quota/epoch = {sum(CFG["quota"].values()):,} mẫu {CFG["quota"]} -> Steps/epoch = {steps_per_ep} | Tổng ngân sách cố định = {CFG["epochs"] * steps_per_ep:,} steps')


train_ds pool: 6777 cặp ({'lolv2': (639, '0.92GB'), 'huawei': (2380, 'lazy'), 'blur': (1858, 'lazy'), 'nikon': (1000, '3.69GB'), 'lolv2syn': (900, '0.80GB')})
Tổng RAM cache train_ds ≈ 5.40 GB | val_ds ≈ 0.49 GB | RAM hệ thống đang dùng ≈ 8.26 GB
Quota/epoch = 4,800 mẫu {'lolv2': 1278, 'huawei': 1122, 'blur': 1600, 'nikon': 400, 'lolv2syn': 400} -> Steps/epoch = 600 | Tổng ngân sách cố định = 42,000 steps


## 🌟 Kiến trúc Đề xuất Duy nhất của Đồ án: `DarkIR-Edge (C=24)` (Cell 8)
Cell 8 dưới đây cài đặt **độc lập và trọn vẹn** mô hình đề xuất duy nhất **`DarkIR-Edge`** ($C=24$, `1.924.038` tham số), kết hợp xương sống **DarkIR (CVPR 2025)** với **2 cải tiến từ bài báo vệ tinh**:
1. **Vận dụng HVI-CIDNet (CVPR 2025) & Lý thuyết Retinex — Phối hợp Hiệp đồng (The Golden Synergy):**
   - **Tại đầu vào mô hình (`extract_imax_prior` + `self.intro` trong Cell 8):** Trích xuất bản đồ độ rọi **Max-RGB Illumination Prior** $I_{\max}(x, y) = \max_{c \in \{R,G,B\}} I_c(x, y)$ lấy cảm hứng từ ý tưởng phân tách cường độ sáng $I$ khỏi mặt phẳng sắc độ trong không gian HVI, ghép trực tiếp thành kênh thứ 4: $X_{\text{in}} = [I_R, I_G, I_B, I_{\max}] \in \mathbb{R}^{B \times 4 \times H \times W}$.
   - **Tại hàm mất mát (`CompositeLoss` trong Cell 11):** Kết hợp **Cosine Color Loss ($\mathcal{L}_{\text{color}}$)** kèm mặt nạ che vùng tối sâu (`mask = gt.amax(1) >= 5/255`) để phạt góc lệch vector màu RGB mà không phải dùng biến đổi lượng giác $\sin/\cos/\arctan$ nhạy cảm đạo hàm của nhánh HVI đầy đủ.
2. **Kế thừa Edge-LLIE (CVPRW 2024) — Cổng lọc nhiễu `GatedSkip`:**
   - Đặt trên cả 3 đường nối tắt từ Encoder sang Decoder: $F_{\text{gated}} = x_{\text{skip}} + \text{Conv}_{1\times1}(x_{\text{skip}}) \odot \sigma(\text{Conv}_{1\times1}(x_{\text{skip}}))$.

In [6]:
# ===== CELL 8: KIẾN TRÚC MÔ HÌNH ĐỀ XUẤT CHÍNH CỦA ĐỒ ÁN — DarkIREdge (C=24) =====
# (Được tách riêng biệt trong cell này để đối chiếu 1:1 với Tài liệu 03)

class LayerNorm2d(nn.Module):
    def __init__(self, c, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(c))
        self.bias = nn.Parameter(torch.zeros(c))
        self.eps = eps

    def forward(self, x):
        x32 = x.float()
        mu = x32.mean(1, keepdim=True)
        var = (x32 - mu).pow(2).mean(1, keepdim=True)
        y = (x32 - mu) / torch.sqrt(var + self.eps)
        return (y * self.weight.view(1, -1, 1, 1) + self.bias.view(1, -1, 1, 1)).to(x.dtype)

class SimpleGate(nn.Module):
    def forward(self, x):
        x1, x2 = x.chunk(2, dim=1)
        return x1 * x2

class FreMLP(nn.Module):
    '''
    [DarkIR CVPR 2025 - Nhánh Tần Số Fourier]
    Điều biến biên độ phổ Fourier (Amplitude) để nâng sáng toàn cục, giữ nguyên pha (Phase).
    Nâng cấp kỹ thuật số học: Chạy ở FP32 và nhân trực tiếp với Phasor đơn vị (xf / mag),
    loại bỏ hoàn toàn hàm lượng giác atan2/cos/sin để chống bùng nổ đạo hàm dưới AMP FP16.
    '''
    def __init__(self, nc, expand=2, norm='backward', eps=1e-6):
        super().__init__()
        self.norm, self.eps = norm, eps
        self.process1 = nn.Sequential(
            nn.Conv2d(nc, expand * nc, 1, 1, 0),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Conv2d(expand * nc, nc, 1, 1, 0)
        )

    def forward(self, x):
        H, W = x.shape[-2:]
        with torch.autocast(device_type=x.device.type, enabled=False):  # Bắt buộc tính cuFFT ở FP32
            xf = torch.fft.rfft2(x.float(), norm=self.norm)
            mag = torch.sqrt(xf.real ** 2 + xf.imag ** 2 + self.eps)
            mag_out = self.process1(mag)
            yf = xf * (mag_out / mag)                                   # xf / mag chính là Phasor đơn vị e^(j*P)
            y = torch.fft.irfft2(yf, s=(H, W), norm=self.norm)
        return y.to(x.dtype)

class Branch(nn.Module):
    '''Nhánh tích chập chiều sâu giãn nở (Dilated Depthwise Convolution) của DarkIR'''
    def __init__(self, c, DW_Expand, dilation=1):
        super().__init__()
        self.dw_channel = DW_Expand * c
        self.branch = nn.Sequential(
            nn.Conv2d(self.dw_channel, self.dw_channel, kernel_size=3, padding=dilation,
                      stride=1, groups=self.dw_channel, bias=True, dilation=dilation)
        )
    def forward(self, x):
        return self.branch(x)

class EBlock(nn.Module):
    '''Khối Encoder chuẩn DarkIR (DarkIR/archs/arch_model.py:141-203): Tích hợp SCA + SimpleGate + FreMLP'''
    def __init__(self, c, DW_Expand=2, dilations=(1,), extra_depth_wise=True, use_fft=True):
        super().__init__()
        self.dw_channel = DW_Expand * c
        self.extra_conv = nn.Conv2d(c, c, kernel_size=3, padding=1, stride=1, groups=c, bias=True, dilation=1) if extra_depth_wise else nn.Identity()
        self.conv1 = nn.Conv2d(c, self.dw_channel, kernel_size=1, padding=0, stride=1, groups=1, bias=True)
        self.branches = nn.ModuleList([Branch(c, DW_Expand, dilation=d) for d in dilations])
        self.sca = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(self.dw_channel // 2, self.dw_channel // 2, kernel_size=1, padding=0, stride=1, groups=1, bias=True)
        )
        self.sg1 = SimpleGate()
        self.conv3 = nn.Conv2d(self.dw_channel // 2, c, kernel_size=1, padding=0, stride=1, groups=1, bias=True)
        self.norm1 = LayerNorm2d(c)
        self.norm2 = LayerNorm2d(c)
        self.freq = FreMLP(nc=c, expand=2) if use_fft else None
        self.gamma = nn.Parameter(torch.zeros((1, c, 1, 1)), requires_grad=True)
        self.beta = nn.Parameter(torch.zeros((1, c, 1, 1)), requires_grad=True)

    def forward(self, inp):
        x = self.norm1(inp)
        x = self.conv1(self.extra_conv(x))
        z = sum(branch(x) for branch in self.branches)
        z = self.sg1(z)
        x = self.sca(z) * z
        x = self.conv3(x)
        y = inp + self.beta * x
        # Bước 2: Điều biến nhân đặc trưng miền tần số (y * x_freq)
        x_step2 = self.norm2(y)
        if self.freq is not None:
            x_freq = self.freq(x_step2)
            x = y * x_freq
            x = y + x * self.gamma
        else:
            x = y
        return x

class DBlock(nn.Module):
    '''Khối Decoder chuẩn DarkIR (DarkIR/archs/arch_model.py:72-140): Cụm tích chập đa tỷ lệ Di-SpAM d=[1, 4, 9]'''
    def __init__(self, c, DW_Expand=2, FFN_Expand=2, dilations=(1, 4, 9), extra_depth_wise=True):
        super().__init__()
        self.dw_channel = DW_Expand * c
        self.conv1 = nn.Conv2d(c, self.dw_channel, kernel_size=1, padding=0, stride=1, groups=1, bias=True)
        self.extra_conv = nn.Conv2d(self.dw_channel, self.dw_channel, kernel_size=3, padding=1, stride=1, groups=c, bias=True, dilation=1) if extra_depth_wise else nn.Identity()
        self.branches = nn.ModuleList([Branch(self.dw_channel, DW_Expand=1, dilation=d) for d in dilations])
        self.sca = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(self.dw_channel // 2, self.dw_channel // 2, kernel_size=1, padding=0, stride=1, groups=1, bias=True)
        )
        self.sg1 = SimpleGate()
        self.sg2 = SimpleGate()
        self.conv3 = nn.Conv2d(self.dw_channel // 2, c, kernel_size=1, padding=0, stride=1, groups=1, bias=True)
        ffn_channel = FFN_Expand * c
        self.conv4 = nn.Conv2d(c, ffn_channel, kernel_size=1, padding=0, stride=1, groups=1, bias=True)
        self.conv5 = nn.Conv2d(ffn_channel // 2, c, kernel_size=1, padding=0, stride=1, groups=1, bias=True)
        self.norm1 = LayerNorm2d(c)
        self.norm2 = LayerNorm2d(c)
        self.gamma = nn.Parameter(torch.zeros((1, c, 1, 1)), requires_grad=True)
        self.beta = nn.Parameter(torch.zeros((1, c, 1, 1)), requires_grad=True)

    def forward(self, inp):
        x = self.norm1(inp)
        x = self.extra_conv(self.conv1(x))
        z = sum(branch(x) for branch in self.branches)
        z = self.sg1(z)
        x = self.sca(z) * z
        x = self.conv3(x)
        y = inp + self.beta * x
        # Bước 2: FFN khôi phục chi tiết không gian
        x = self.conv4(self.norm2(y))
        x = self.sg2(x)
        x = self.conv5(x)
        return y + x * self.gamma

# =========================================================================
# CẢI TIẾN VỆ TINH 1 (Lấy cảm hứng từ HVI-CIDNet CVPR 2025 & Lý thuyết Retinex):
# Trích xuất bản đồ độ rọi cực đại Max-RGB Illumination Prior: I_max = max(R, G, B)
# và ghép vào ảnh RGB đầu vào tạo thành Tensor 4 kênh [R, G, B, I_max].
# (Kết hợp cùng hàm Cosine Color Loss ở Cell 11 để bảo toàn sắc độ mà tốn 0 tham số ẩn).
# =========================================================================
def extract_imax_prior(rgb_tensor):
    '''Input: rgb_tensor (B, 3, H, W) -> Output: [R, G, B, I_max] (B, 4, H, W)'''
    i_max = rgb_tensor.amax(dim=1, keepdim=True)   # Kênh độ rọi I_max ∈ R^(B x 1 x H x W)
    return torch.cat([rgb_tensor, i_max], dim=1)

# =========================================================================
# CẢI TIẾN VỆ TINH 2 (Kế thừa từ Edge-LLIE CVPRW 2024):
# Cổng lọc nhiễu dư Residual Gated Skip Connection trên 3 tầng nối tắt Encoder -> Decoder
# F_gated = x_skip + Conv1x1(x_skip) * Sigmoid(Conv1x1(x_skip))
# =========================================================================
class GatedSkip(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.f = nn.Conv2d(c, c, 1)
        self.g = nn.Conv2d(c, c, 1)
    def forward(self, x):
        return x + self.f(x) * torch.sigmoid(self.g(x))

def pad8(x):
    '''Đệm phản xạ (Reflection Padding) lên bội số của 8 để hỗ trợ ảnh kích thước bất kỳ'''
    H, W = x.shape[-2:]
    ph, pw = (8 - H % 8) % 8, (8 - W % 8) % 8
    return F.pad(x, (0, pw, 0, ph), mode='reflect') if (ph or pw) else x

class DarkIREdge(nn.Module):
    '''
    KIẾN TRÚC ĐỀ XUẤT DUY NHẤT CỦA ĐỒ ÁN: DarkIR-Edge (width=24 -> 1,924,038 tham số)
    - Bảo toàn 100% cấu trúc bất đối xứng của DarkIR: Encoder [1, 2, 3] (EBlock + FreMLP),
      Bottleneck [2 EBlock + 2 DBlock + Side Supervision 8C->3], Decoder [3, 1, 1] (DBlock + Di-SpAM).
    - Tích hợp 2 cải tiến vệ tinh:
      (1) use_imax=True: Ghép kênh tiên nghiệm độ rọi Max-RGB (HVI/Retinex Inspiration) -> Intro Conv nhận 4 kênh.
      (2) use_gate=True: Cổng lọc nhiễu Residual Gated Skip Connection (Edge-LLIE) tại 3 tầng skip.
    '''
    def __init__(self, width=24, enc_blk_nums=(1, 2, 3), dec_blk_nums=(3, 1, 1),
                 middle_blk_num_enc=2, middle_blk_num_dec=2, dilations=(1, 4, 9),
                 extra_depth_wise=True, use_imax=True, use_gate=True, use_fft=True):
        super().__init__()
        self.use_imax = use_imax
        in_ch = 4 if use_imax else 3
        self.intro = nn.Conv2d(in_ch, width, kernel_size=3, padding=1, stride=1, bias=True)
        self.ending = nn.Conv2d(width, 3, kernel_size=3, padding=1, stride=1, bias=True)
        self.encoders, self.downs = nn.ModuleList(), nn.ModuleList()
        self.decoders, self.ups, self.gates = nn.ModuleList(), nn.ModuleList(), nn.ModuleList()

        chan = width
        for num in enc_blk_nums:
            self.encoders.append(nn.Sequential(*[EBlock(chan, extra_depth_wise=extra_depth_wise, use_fft=use_fft) for _ in range(num)]))
            self.downs.append(nn.Conv2d(chan, 2 * chan, 2, 2))
            chan *= 2

        self.middle_blks_enc = nn.Sequential(*[EBlock(chan, extra_depth_wise=extra_depth_wise, use_fft=use_fft) for _ in range(middle_blk_num_enc)])
        self.middle_blks_dec = nn.Sequential(*[DBlock(chan, dilations=dilations, extra_depth_wise=extra_depth_wise) for _ in range(middle_blk_num_dec)])
        self.side_out = nn.Conv2d(chan, 3, kernel_size=3, stride=1, padding=1)

        for num in dec_blk_nums:
            self.ups.append(nn.Sequential(nn.Conv2d(chan, chan * 2, 1, bias=False), nn.PixelShuffle(2)))
            chan //= 2
            self.gates.append(GatedSkip(chan) if use_gate else nn.Identity())
            self.decoders.append(nn.Sequential(*[DBlock(chan, dilations=dilations, extra_depth_wise=extra_depth_wise) for _ in range(num)]))

    def forward(self, inp, return_side=False):
        H, W = inp.shape[-2:]
        x_pad = pad8(inp)
        # Vận dụng Cải tiến Vệ tinh 1 (Max-RGB Illumination Prior từ cảm hứng HVI/Retinex)
        x_in = extract_imax_prior(x_pad) if self.use_imax else x_pad
        x = self.intro(x_in)

        skips = []
        for encoder, down in zip(self.encoders, self.downs):
            x = encoder(x)
            skips.append(x)
            x = down(x)

        x_light = self.middle_blks_enc(x)
        out_side = self.side_out(x_light) if return_side else None
        x = self.middle_blks_dec(x_light)
        x = x + x_light   # Kết nối tắt nội bộ tại Bottleneck (chuẩn DarkIR.py dòng 94)

        for decoder, up, gate, skip in zip(self.decoders, self.ups, self.gates, skips[::-1]):
            x = up(x)
            # Vận dụng Cải tiến Vệ tinh 2 (Residual Gated Skip Connection từ Edge-LLIE)
            x = x + gate(skip)
            x = decoder(x)

        x = self.ending(x) + x_pad
        out = x[..., :H, :W]
        if return_side and out_side is not None:
            out_side = out_side[..., :H // 8, :W // 8]
        return (out, out_side) if return_side else out

def n_params(m):
    return sum(p.numel() for p in m.parameters())

print(f'✔ Đã định nghĩa DarkIREdge (C=24) -> Tổng số tham số: {n_params(DarkIREdge(width=24)):,} params')

✔ Đã định nghĩa DarkIREdge (C=24) -> Tổng số tham số: 1,924,038 params


In [7]:
# ===== CELL 9 (Tách riêng): Kiến trúc Baseline Đối chuẩn — Standard U-Net (base=32, ~1.93M params) =====
# Mô hình này chỉ dùng làm mốc đối chuẩn (Baseline) cùng quy mô tham số để so sánh với DarkIR-Edge.

def _cb(i, o, g=4):
    return nn.Sequential(
        nn.Conv2d(i, o, 3, padding=1), nn.GroupNorm(g, o), nn.ReLU(True),
        nn.Conv2d(o, o, 3, padding=1), nn.GroupNorm(g, o), nn.ReLU(True)
    )

class UNetBaseline(nn.Module):
    '''Standard U-Net tích chập thuần (Plain Conv2d), cùng độ sâu 3 tầng downsample & Global Residual'''
    def __init__(self, base=32):
        super().__init__()
        b = base
        self.e1, self.e2, self.e3, self.mid = _cb(3, b), _cb(b, 2 * b), _cb(2 * b, 4 * b), _cb(4 * b, 8 * b)
        self.u3 = nn.ConvTranspose2d(8 * b, 4 * b, 2, 2)
        self.u2 = nn.ConvTranspose2d(4 * b, 2 * b, 2, 2)
        self.u1 = nn.ConvTranspose2d(2 * b, b, 2, 2)
        self.d3, self.d2, self.d1 = _cb(8 * b, 4 * b), _cb(4 * b, 2 * b), _cb(2 * b, b)
        self.out = nn.Conv2d(b, 3, 1)

    def forward(self, x, return_side=False):
        H, W = x.shape[-2:]
        x_pad = pad8(x)
        e1 = self.e1(x_pad)
        e2 = self.e2(F.max_pool2d(e1, 2))
        e3 = self.e3(F.max_pool2d(e2, 2))
        m  = self.mid(F.max_pool2d(e3, 2))
        d3 = self.d3(torch.cat([self.u3(m), e3], 1))
        d2 = self.d2(torch.cat([self.u2(d3), e2], 1))
        d1 = self.d1(torch.cat([self.u1(d2), e1], 1))
        out = (self.out(d1) + x_pad)[..., :H, :W]
        return (out, None) if return_side else out

print(f'✔ Đã định nghĩa UNetBaseline (base={CFG["unet_base"]}) -> Tổng số tham số: {n_params(UNetBaseline(CFG["unet_base"])):,} params')

✔ Đã định nghĩa UNetBaseline (base=32) -> Tổng số tham số: 1,928,483 params


In [8]:
# ===== CELL 10: Kiểm chứng số lượng tham số từng tầng & Smoke test kích thước ảnh lẻ =====
edge = DarkIREdge(width=24, use_imax=True, use_gate=True)
darkir_m_ref = DarkIREdge(width=32, use_imax=False, use_gate=False)
unet = UNetBaseline(base=CFG['unet_base'])
nofft = DarkIREdge(width=24, use_fft=False)

print(f'1. DarkIR-Edge (C=24, ĐỀ XUẤT CHÍNH): {n_params(edge):,} tham số (khớp chuẩn 1,924,038 trong File 03)')
print(f'2. Standard U-Net (base={CFG["unet_base"]}):          {n_params(unet):,} tham số (đối chuẩn cùng quy mô ~1.93M)')
print(f'3. DarkIR-m gốc (C=32, tham chiếu):   {n_params(darkir_m_ref):,} tham số (khớp chuẩn 3,321,638 của CVPR 2025)')

print(f'4. DarkIR-Edge bỏ FFT/FreMLP (C=24, ablation): {n_params(nofft):,} tham số (ít hơn bản đầy đủ {n_params(DarkIREdge(width=24)) - n_params(nofft):,} tham số do không còn FreMLP)')

x_test = torch.rand(1, 3, 61, 83)
for name, m in [('DarkIR-Edge', edge), ('DarkIR-Edge no-FFT', nofft), ('Standard U-Net', unet)]:
    out, side = m(x_test, return_side=True)
    out.mean().backward()
    print(f'  [OK] {name}: input {tuple(x_test.shape)} -> output {tuple(out.shape)} | side {None if side is None else tuple(side.shape)}')
assert out.shape == x_test.shape
del edge, darkir_m_ref, unet, nofft

1. DarkIR-Edge (C=24, ĐỀ XUẤT CHÍNH): 1,924,038 tham số (khớp chuẩn 1,924,038 trong File 03)
2. Standard U-Net (base=32):          1,928,483 tham số (đối chuẩn cùng quy mô ~1.93M)
3. DarkIR-m gốc (C=32, tham chiếu):   3,321,638 tham số (khớp chuẩn 3,321,638 của CVPR 2025)
4. DarkIR-Edge bỏ FFT/FreMLP (C=24, ablation): 1,495,422 tham số (ít hơn bản đầy đủ 428,616 tham số do không còn FreMLP)
  [OK] DarkIR-Edge: input (1, 3, 61, 83) -> output (1, 3, 61, 83) | side (1, 3, 7, 10)
  [OK] DarkIR-Edge no-FFT: input (1, 3, 61, 83) -> output (1, 3, 61, 83) | side (1, 3, 7, 10)
  [OK] Standard U-Net: input (1, 3, 61, 83) -> output (1, 3, 61, 83) | side None


In [9]:
# ===== CELL 11: Hàm mất mát tổng hợp CompositeLoss (gồm cả Cosine Color Loss), EMA & Metrics =====
def _gauss_win(size=11, sigma=1.5, ch=3):
    g = torch.arange(size).float() - size // 2
    g = torch.exp(-g ** 2 / (2 * sigma ** 2))
    g /= g.sum()
    return (g[:, None] * g[None, :]).expand(ch, 1, size, size).contiguous()

def ssim_torch(x, y, win):
    ch = x.shape[1]; w = win.to(x.device)
    mu_x, mu_y = F.conv2d(x, w, groups=ch), F.conv2d(y, w, groups=ch)
    sxx = F.conv2d(x * x, w, groups=ch) - mu_x ** 2
    syy = F.conv2d(y * y, w, groups=ch) - mu_y ** 2
    sxy = F.conv2d(x * y, w, groups=ch) - mu_x * mu_y
    C1, C2 = 0.01 ** 2, 0.03 ** 2
    return (((2 * mu_x * mu_y + C1) * (2 * sxy + C2)) / ((mu_x ** 2 + mu_y ** 2 + C1) * (sxx + syy + C2))).mean()

class CompositeLoss(nn.Module):
    '''
    Hàm mất mát đa nhiệm chuẩn Tài liệu 03 (Phần IV.1):
    L_total = L1(pred, gt)
            + w_side  * L1(side, avg_pool2d(gt, 8))             [Deep Supervision tại Bottleneck]
            + w_ssim  * (1 - SSIM(pred, gt))                    [Bảo toàn cấu trúc bề mặt]
            + w_color * CosineColorLoss(pred, gt, mask>=5/255)  [Chống lệch màu theo tinh thần HVI/Retinex]
    '''
    def __init__(self, w, use_color=True):
        super().__init__()
        self.w, self.use_color = w, use_color
        self.win = _gauss_win()

    def forward(self, pred, gt, side=None):
        pred, gt = pred.float(), gt.float()
        parts = {'l1': F.l1_loss(pred, gt)}
        parts['ssim'] = self.w['ssim'] * (1.0 - ssim_torch(pred.clamp(0, 1), gt, self.win))
        if side is not None:
            gt_down = F.avg_pool2d(gt, kernel_size=8, stride=8)
            parts['side'] = self.w['side'] * F.l1_loss(side.float(), gt_down)
        if self.use_color:
            # Cosine Color Loss kèm mặt nạ che các pixel tối sâu (< 5/255) để chống bùng nổ gradient góc màu
            cos = F.cosine_similarity(pred, gt, dim=1, eps=1e-6)
            mask = (gt.amax(1) >= 5.0 / 255.0).float()
            parts['color'] = self.w['color'] * (1.0 - (cos * mask).sum() / mask.sum().clamp(min=1.0))
        return sum(parts.values()), parts

class EMA:
    def __init__(self, model, decay):
        self.m = copy.deepcopy(model).eval()
        self.decay, self.n = decay, 0

    @torch.no_grad()
    def update(self, model):
        self.n += 1
        d = min(self.decay, (1 + self.n) / (10 + self.n))
        for pe, p in zip(self.m.parameters(), model.parameters()):
            pe.mul_(d).add_(p.detach(), alpha=1 - d)
        for be, b in zip(self.m.buffers(), model.buffers()):
            be.copy_(b)

def psnr_u8(a, b):
    mse = np.mean((a.astype(np.float64) - b.astype(np.float64)) ** 2)
    return 99.0 if mse == 0 else 10.0 * math.log10(255.0 ** 2 / mse)

@torch.no_grad()
def predict_u8(model, img_u8):
    x = to_tensor(img_u8).unsqueeze(0).to(DEV)
    with torch.autocast(device_type=DEV.type, dtype=torch.float16, enabled=USE_AMP):
        y = model(x)
    return (y.float().clamp(0, 1)[0].permute(1, 2, 0).cpu().numpy() * 255.0).round().astype(np.uint8)

@torch.no_grad()
def val_psnr(model, ds):
    model.eval()
    return float(np.mean([psnr_u8(predict_u8(model, lo), hi) for lo, hi in ds.cache]))

def val_psnr_macro(model, ds_by_dom):
    '''PSNR macro = trung bình PSNR của từng miền Val (LOLv2, Huawei, Blur; mỗi miền 50 ảnh = 150 ảnh Val cân bằng ở v3).'''
    per_dom = {d: val_psnr(model, ds) for d, ds in ds_by_dom.items()}
    return float(np.mean(list(per_dom.values()))), per_dom

## 🚀 Huấn luyện Mô hình (Tách riêng Cell cho `DarkIR-Edge` và Cell cho `Standard U-Net`)
- **v4 — Mở rộng đa miền + Quota Sampler (`4.800` mẫu/epoch = `600` bước/epoch) + Chạy cố định 70 epoch (`42.000 steps`, `T_max = 42.000`):** bảo toàn tuyệt đối lượng luyện tăng sáng của v3 (`1.278` LOLv2-Real + `1.122` LSRW-Huawei/epoch) và cộng thêm dữ liệu mới (`1.600` Blur + `400` Nikon + `400` LOLv2-Syn/epoch). Chỉ huấn luyện **`DarkIR-Edge`** và **`Standard U-Net`** (2 seed `[42, 123]`; bỏ `no-FFT`). Trọng số đánh giá chính thức: **`final.pt` (EMA tại epoch 70)**.
- **Hướng dẫn chạy 2 phiên theo seed (chỉnh trực tiếp tại Cell 14 và Cell 15, không cần sửa Cell 1):** Phiên 1 đặt `SEEDS_TO_RUN_DARKIR_EDGE = [42]` (Cell 14) và `SEEDS_TO_RUN_UNET = [42]` (Cell 15); Phiên 2 đặt `= [123]` ở cả 2 cell đó, sau khi train xong chạy thẳng các cell đánh giá (tự động đọc cả `[42, 123]` từ `CFG`).
- **Về cơ chế `EarlyStopping` trong `train_model` (Cell 12) — mô tả cũ của v1:**
  - Toàn bộ khối kiểm tra `EarlyStopping` được đóng khung rõ ràng bằng chú thích `=== [KHỐI EARLY STOPPING] ===` ngay bên trong `train_model`.
  - **Trạng thái mặc định hiện tại:** Đang **TẮT (`CFG['use_early_stopping'] = False`)** để mô hình chạy trọn vẹn số epochs quy định cho lịch trình `CosineAnnealingLR` hội tụ mượt về $10^{-6}$ và luôn lưu thêm `best.pt` song song với `final.pt`. Khi cần kích hoạt cơ chế dừng sớm cho thử nghiệm phụ, đặt `CFG['use_early_stopping'] = True`.
- **Thứ tự chạy v4:** Cell 14 (DarkIR-Edge) → Cell 15 (U-Net, 2 seed) → Cell ablation (bỏ FFT, 2 seed).
- **Thứ tự ưu tiên chạy (v1):**
  - **Cell 13:** Đo tốc độ thực tế (`probe_speed`) 200 bước đầu để biết chính xác số phút/run trên GPU Colab hiện tại.
  - **Cell 14 (ƯU TIÊN SỐ 1):** Chạy huấn luyện riêng cho mô hình chính **`DarkIR-Edge (C=24)`**.
  - **Cell 15 (TÙY CHỌN):** Chạy huấn luyện riêng cho **`Standard U-Net`** đối chuẩn.

In [10]:
# ===== CELL 12: Hàm huấn luyện cốt lõi (AMP + Gradient Clipping + EMA + Auto-Resume + Khối Early Stopping rõ ràng) =====
def build(tag):
    if tag == 'darkir_edge': return DarkIREdge(width=24)
    if tag == 'darkir_edge_nofft': return DarkIREdge(width=24, use_fft=False)   # Ablation: bỏ nhánh FFT/FreMLP, giữ nguyên mọi thứ khác
    if tag == 'unet':        return UNetBaseline(CFG['unet_base'])
    raise ValueError(f'Không nhận diện được mô hình: {tag}')

def train_model(tag, seed):
    out_dir = Path(CFG['ckpt_dir']) / f'{tag}_seed{seed}'
    out_dir.mkdir(parents=True, exist_ok=True)

    # Kiểm tra trạng thái huấn luyện cũ & cờ FORCE_RETRAIN (v3: đã vô hiệu hóa khối [AUTO-CONTINUE] của Early Stopping; chỉ dừng khi đủ CFG['epochs'])
    if (out_dir / 'final.pt').exists() and not CFG.get('force_retrain', False):
        ck_fin = torch.load(out_dir / 'final.pt', map_location='cpu', weights_only=False)
        ep_done = ck_fin.get('epochs_run', CFG['epochs'])
        print(f'[SKIP] {tag} (seed={seed}) trong phiên bản "{CFG["version"]}" đã có final.pt (epoch {ep_done}/{CFG["epochs"]}) -> bỏ qua.')
        print(f'       (Để huấn luyện lại, đổi EXP_VERSION sang phiên bản mới hoặc đặt FORCE_RETRAIN=True).')
        return
    elif (out_dir / 'final.pt').exists() and CFG.get('force_retrain', False):
        print(f'[RETRAIN] Phát hiện final.pt cũ nhưng FORCE_RETRAIN=True -> Huấn luyện lại từ đầu cho {tag} (seed={seed})!')

    # Khóa cứng hạt giống trước khi khởi tạo mô hình và DataLoader -> tái lập 100%
    set_seed(seed)
    model = build(tag).to(DEV)
    ema = EMA(model, CFG['ema_decay'])
    crit = CompositeLoss(CFG['loss_w']).to(DEV)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], betas=CFG['betas'], weight_decay=CFG['weight_decay'])
    loader = make_train_loader(seed)
    spe = len(loader) if CFG['max_steps_per_epoch'] is None else min(len(loader), CFG['max_steps_per_epoch'])
    total_steps = CFG['epochs'] * spe
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=total_steps, eta_min=CFG['lr_min'])
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)

    history, start_ep = [], 0
    best_val_psnr = -1.0
    early_stop_wait = 0
    es_ref = -1.0   # Mốc val PSNR dùng cho Early Stopping (chỉ cập nhật khi cải thiện > min_delta)

    # Auto-Resume: chỉ resume nếu có last.pt và không bật FORCE_RETRAIN
    if (out_dir / 'last.pt').exists() and not CFG.get('force_retrain', False):
        ck = torch.load(out_dir / 'last.pt', map_location=DEV, weights_only=False)
        model.load_state_dict(ck['model']); ema.m.load_state_dict(ck['ema'])
        opt.load_state_dict(ck['opt']); sched.load_state_dict(ck['sched']); scaler.load_state_dict(ck['scaler'])
        history, start_ep, ema.n = ck['history'], ck['epoch'], ck['ema_n']
        best_val_psnr = ck.get('best_val_psnr', -1.0)
        early_stop_wait = 0 if start_ep <= CFG.get('early_stop_min_epochs', 0) else ck.get('early_stop_wait', 0)
        es_ref = ck.get('es_ref', best_val_psnr)
        print(f'[RESUME] Khôi phục {tag} (seed={seed}) từ epoch {start_ep}/{CFG["epochs"]}')

    print(f'=== Bắt đầu huấn luyện {tag} (seed={seed}) [Version: {CFG["version"]}] | {n_params(model):,} params | {spe} steps/ep × {CFG["epochs"]} ep = {total_steps:,} steps ===')
    t0 = time.time()

    if DEV.type == 'cuda':
        torch.cuda.reset_peak_memory_stats()
    for ep in range(start_ep, CFG['epochs']):
        if hasattr(loader.sampler, 'set_epoch'):
            loader.sampler.set_epoch(ep)
        model.train()
        run_metrics, n_batches = {}, 0
        for it, (lo, hi) in enumerate(loader):
            if it >= spe: break
            lo, hi = lo.to(DEV, non_blocking=True), hi.to(DEV, non_blocking=True)
            with torch.autocast(device_type=DEV.type, dtype=torch.float16, enabled=USE_AMP):
                pred, side = model(lo, return_side=True)
            loss, parts = crit(pred, hi, side)
            if not torch.isfinite(loss):
                print(f'WARN: Loss không hữu hạn tại ep {ep+1} step {it} -> bỏ qua bước này')
                opt.zero_grad(set_to_none=True); continue

            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG['grad_clip'])
            scaler.step(opt)
            scaler.update()
            sched.step()
            ema.update(model)

            if ep == start_ep and it == 0 and DEV.type == 'cuda':
                vram_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
                ram_gb = (psutil.virtual_memory().used / (1024 ** 3)) if psutil is not None else 0.0
                print(f'  [Memory Step 0] Peak VRAM = {vram_mb:.1f} MB | System RAM used = {ram_gb:.2f} GB')
            run_metrics['loss'] = run_metrics.get('loss', 0.0) + loss.item()
            for k, v in parts.items():
                run_metrics[k] = run_metrics.get(k, 0.0) + v.item()
            n_batches += 1

        rec = {'epoch': ep + 1, 'lr': sched.get_last_lr()[0], **{k: v / max(n_batches, 1) for k, v in run_metrics.items()}}
        if (ep + 1) % CFG['val_interval'] == 0 or (ep + 1) == CFG['epochs']:
            cur_val, per_dom = val_psnr_macro(ema.m, val_by_dom)   # PSNR macro các miền Val -> chỉ số theo dõi
            rec['val_psnr_ema'] = cur_val
            rec.update({f'val_psnr_{d}': v for d, v in per_dom.items()})

            # Luôn lưu lại best.pt khi đạt đỉnh Val PSNR mới để phục vụ phân tích phụ
            if cur_val > best_val_psnr:
                best_val_psnr = cur_val
                torch.save({'ema': ema.m.state_dict(), 'model': model.state_dict(), 'epoch': ep + 1, 'val_psnr': cur_val}, out_dir / 'best.pt')
            # Chỉ bắt đầu đếm early_stop_wait SAU khi đã vượt qua early_stop_min_epochs (vd: từ epoch 45 trở đi)
            if (ep + 1) <= CFG['early_stop_min_epochs']:
                es_ref = max(es_ref, cur_val)
                early_stop_wait = 0
            elif cur_val > es_ref + CFG['early_stop_min_delta']:
                es_ref = cur_val
                early_stop_wait = 0
            else:
                early_stop_wait += CFG['val_interval']

            torch.save({
                'model': model.state_dict(), 'ema': ema.m.state_dict(),
                'opt': opt.state_dict(), 'sched': sched.state_dict(), 'scaler': scaler.state_dict(),
                'history': history + [rec], 'epoch': ep + 1, 'ema_n': ema.n,
                'best_val_psnr': best_val_psnr, 'early_stop_wait': early_stop_wait, 'es_ref': es_ref
            }, out_dir / 'last.pt')

            # =====================================================================
            # === [KHỐI EARLY STOPPING — ĐANG TẮT Ở v3 (use_early_stopping=False) ĐỂ CHẠY ĐỦ 70 EPOCH CỐ ĐỊNH] ===
            # Chỉ dựa trên Val; không dừng trước early_stop_min_epochs; quy tắc cố định cho mọi mô hình.
            # =====================================================================
            if CFG['use_early_stopping'] and (ep + 1) >= CFG['early_stop_min_epochs'] and early_stop_wait >= CFG['early_stop_patience']:
                print(f'[EARLY STOPPING] Dừng sớm tại epoch {ep+1}: val PSNR macro không tăng quá {CFG["early_stop_min_delta"]} dB sau {early_stop_wait} epochs (Best={best_val_psnr:.2f} dB).')
                history.append(rec)
                break
            # =====================================================================

        history.append(rec)
        if 'val_psnr_ema' in rec or ep == start_ep:
            elapsed = (time.time() - t0) / 60.0
            print(f'Epoch {ep+1:3d}/{CFG["epochs"]} | Loss: {rec.get("loss", float("nan")):.4f} | LR: {rec["lr"]:.2e} | Val PSNR (EMA): {rec.get("val_psnr_ema", float("nan")):.2f} dB (Best: {best_val_psnr:.2f}) | {elapsed:.1f} phút')

    pd.DataFrame(history).to_csv(out_dir / 'history.csv', index=False)
    torch.save({
        'ema': ema.m.state_dict(), 'model': model.state_dict(),
        'tag': tag, 'seed': seed, 'best_val_psnr': best_val_psnr, 'epochs_run': history[-1]['epoch'],
        'cfg': {k: v for k, v in CFG.items() if k != 'runs'}
    }, out_dir / 'final.pt')
    print(f'✔ Hoàn tất {tag} (seed={seed}) sau {history[-1]["epoch"]}/{CFG["epochs"]} epoch, trong {(time.time() - t0) / 60.0:.1f} phút -> Đã lưu final.pt & best.pt tại {out_dir}')

In [11]:
# ===== CELL 13: ĐO TỐC ĐỘ THỰC TẾ & NGÂN SÁCH BỘ NHỚ (RAM/VRAM) TRƯỚC KHI HUẤN LUYỆN DÀI =====
def probe_speed(tag, steps=100):
    set_seed(MASTER_SEED)
    if DEV.type == 'cuda':
        torch.cuda.reset_peak_memory_stats()
    m = build(tag).to(DEV).train()
    crit = CompositeLoss(CFG['loss_w']).to(DEV)
    opt = torch.optim.AdamW(m.parameters(), lr=1e-4)
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)
    loader = make_train_loader(MASTER_SEED)
    it_loader = iter(loader)
    steps = 6 if TEST_MODE else steps
    t_start = None
    for i in range(steps):
        if i == 3:
            if DEV.type == 'cuda': torch.cuda.synchronize()
            t_start = time.time()
        try:
            lo, hi = next(it_loader)
        except StopIteration:
            it_loader = iter(loader); lo, hi = next(it_loader)
        lo, hi = lo.to(DEV), hi.to(DEV)
        with torch.autocast(device_type=DEV.type, dtype=torch.float16, enabled=USE_AMP):
            pred, side = m(lo, return_side=True)
        loss, _ = crit(pred, hi, side)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
    if DEV.type == 'cuda': torch.cuda.synchronize()
    elapsed = max(time.time() - t_start, 1e-6)
    ips = (steps - 3) / elapsed
    sec_per_step = 1.0 / max(ips, 1e-6)
    spe = sum(CFG['quota'].values()) // CFG['batch_size']
    total_steps = CFG['epochs'] * spe
    est_min = total_steps / max(ips, 1e-6) / 60.0
    ram_str = (f'{psutil.virtual_memory().used/(1024**3):.2f}/{psutil.virtual_memory().total/(1024**3):.1f} GB ({psutil.virtual_memory().percent}%)' if psutil is not None else 'N/A')
    vram_mb = torch.cuda.max_memory_allocated() / (1024 ** 2) if DEV.type == 'cuda' else 0.0
    print(
        f'[{tag:12s}] {ips:.2f} it/s ({sec_per_step:.3f} s/step) | '
        f'1 run ({total_steps:,} steps = {CFG["epochs"]} ep × {spe} steps/ep): ~{est_min:.1f} phút | '
        f'RAM cache={train_ds.gb:.2f} GB, RAM sys={ram_str} | '
        f'Peak VRAM={vram_mb:.1f} MB'
    )
    del m, crit, opt, scaler

probe_speed('darkir_edge')
if CFG.get('darkir_edge_nofft_seeds'):
    probe_speed('darkir_edge_nofft')
probe_speed('unet')


[darkir_edge ] 3.61 it/s (0.277 s/step) | 1 run (42,000 steps = 70 ep × 600 steps/ep): ~194.1 phút | RAM cache=5.40 GB, RAM sys=9.91/53.0 GB (20.2%) | Peak VRAM=4815.9 MB
[unet        ] 13.75 it/s (0.073 s/step) | 1 run (42,000 steps = 70 ep × 600 steps/ep): ~50.9 phút | RAM cache=5.40 GB, RAM sys=9.94/53.0 GB (20.2%) | Peak VRAM=1471.2 MB


In [12]:
# ===== CELL 14 (ƯU TIÊN SỐ 1): HUẤN LUYỆN MÔ HÌNH ĐỀ XUẤT CHÍNH — DarkIR-Edge (C=24) =====
# Phiên 1: đặt SEEDS_TO_RUN_DARKIR_EDGE = [42] | Phiên 2: đặt SEEDS_TO_RUN_DARKIR_EDGE = [123]
# Không cần chỉnh sửa CELL 1.
# Ví dụ: SEEDS_TO_RUN_DARKIR_EDGE = [42] (Phiên 1) hoặc [123] (Phiên 2)

# Chỉnh trực tiếp danh sách seed cần huấn luyện tại đây ([42] cho Phiên 1, [123] cho Phiên 2, hoặc [42, 123]):
SEEDS_TO_RUN_DARKIR_EDGE = [123]

for seed in SEEDS_TO_RUN_DARKIR_EDGE:
    train_model('darkir_edge', seed=seed)

=== Bắt đầu huấn luyện darkir_edge (seed=123) [Version: v4] | 1,924,038 params | 600 steps/ep × 70 ep = 42,000 steps ===
  [Memory Step 0] Peak VRAM = 4793.4 MB | System RAM used = 9.91 GB
Epoch   1/70 | Loss: 0.2703 | LR: 2.00e-04 | Val PSNR (EMA): nan dB (Best: -1.00) | 2.9 phút
Epoch   5/70 | Loss: 0.2144 | LR: 1.98e-04 | Val PSNR (EMA): 20.68 dB (Best: 20.68) | 14.9 phút
Epoch  10/70 | Loss: 0.2023 | LR: 1.90e-04 | Val PSNR (EMA): 21.07 dB (Best: 21.07) | 29.8 phút
Epoch  15/70 | Loss: 0.1904 | LR: 1.78e-04 | Val PSNR (EMA): 21.14 dB (Best: 21.14) | 44.6 phút
Epoch  20/70 | Loss: 0.1838 | LR: 1.63e-04 | Val PSNR (EMA): 21.19 dB (Best: 21.19) | 59.5 phút
Epoch  25/70 | Loss: 0.1782 | LR: 1.44e-04 | Val PSNR (EMA): 21.18 dB (Best: 21.19) | 74.3 phút
Epoch  30/70 | Loss: 0.1739 | LR: 1.23e-04 | Val PSNR (EMA): 21.36 dB (Best: 21.36) | 89.2 phút
Epoch  35/70 | Loss: 0.1679 | LR: 1.01e-04 | Val PSNR (EMA): 21.57 dB (Best: 21.57) | 104.0 phút
Epoch  40/70 | Loss: 0.1640 | LR: 7.84e-05 | 

In [13]:
# ===== CELL 15 (TÁCH RIÊNG - TÙY CHỌN): HUẤN LUYỆN BASELINE ĐỐI CHUẨN — Standard U-Net =====
# Đặt RUN_UNET_BASELINE = True để huấn luyện Standard U-Net (~1.93M params) phục vụ so sánh đối chuẩn.
# Đặt RUN_UNET_BASELINE = False khi chỉ thực thi huấn luyện DarkIR-Edge.

RUN_UNET_BASELINE = True
# Chỉnh trực tiếp danh sách seed cần huấn luyện tại đây ([42] cho Phiên 1, [123] cho Phiên 2, hoặc [42, 123]):
SEEDS_TO_RUN_UNET = [123]

if RUN_UNET_BASELINE:
    for seed in SEEDS_TO_RUN_UNET:
        train_model('unet', seed=seed)
else:
    print('[SKIP] RUN_UNET_BASELINE = False -> Bỏ qua huấn luyện Standard U-Net.')

=== Bắt đầu huấn luyện unet (seed=123) [Version: v4] | 1,928,483 params | 600 steps/ep × 70 ep = 42,000 steps ===
  [Memory Step 0] Peak VRAM = 1448.3 MB | System RAM used = 9.91 GB
Epoch   1/70 | Loss: 0.1869 | LR: 2.00e-04 | Val PSNR (EMA): nan dB (Best: -1.00) | 0.8 phút
Epoch   5/70 | Loss: 0.1632 | LR: 1.98e-04 | Val PSNR (EMA): 20.58 dB (Best: 20.58) | 4.0 phút
Epoch  10/70 | Loss: 0.1543 | LR: 1.90e-04 | Val PSNR (EMA): 20.82 dB (Best: 20.82) | 7.9 phút
Epoch  15/70 | Loss: 0.1463 | LR: 1.78e-04 | Val PSNR (EMA): 20.92 dB (Best: 20.92) | 11.9 phút
Epoch  20/70 | Loss: 0.1430 | LR: 1.63e-04 | Val PSNR (EMA): 21.14 dB (Best: 21.14) | 15.8 phút
Epoch  25/70 | Loss: 0.1407 | LR: 1.44e-04 | Val PSNR (EMA): 21.30 dB (Best: 21.30) | 19.8 phút
Epoch  30/70 | Loss: 0.1382 | LR: 1.23e-04 | Val PSNR (EMA): 21.37 dB (Best: 21.37) | 23.8 phút
Epoch  35/70 | Loss: 0.1361 | LR: 1.01e-04 | Val PSNR (EMA): 21.61 dB (Best: 21.61) | 27.7 phút
Epoch  40/70 | Loss: 0.1336 | LR: 7.84e-05 | Val PSNR (

## 🔬 Ablation giả thuyết FFT/FreMLP (v2)
Giả thuyết ở v1 — nhánh FFT/FreMLP giúp nâng sáng toàn cục — **chưa được kiểm chứng**. Cell dưới đây huấn luyện `DarkIR-Edge` **bỏ FFT/FreMLP** (`use_fft=False`, mọi thứ khác giữ nguyên: Imax, GatedSkip, side-loss, dữ liệu, loss, Early Stopping) với cùng 2 seed. Kết quả được so sánh ghép cặp với DarkIR-Edge đầy đủ ở Cell đánh giá và báo cáo đúng như thực tế, dù FFT có giúp hay không.

In [14]:
# ===== CELL ABLATION (v2): HUẤN LUYỆN DarkIR-Edge BỎ FFT/FreMLP — kiểm chứng giả thuyết FFT =====
CFG['darkir_edge_nofft_seeds'] = [42, 123]
# RUN_NOFFT_ABLATION = bool(CFG.get('darkir_edge_nofft_seeds'))   # v4: [] -> tự động bỏ qua no-FFT
RUN_NOFFT_ABLATION = True
SEEDS_TO_RUN_NOFFT = [123]
# SEEDS_TO_RUN_NOFFT = CFG['darkir_edge_nofft_seeds']

if RUN_NOFFT_ABLATION:
    for seed in SEEDS_TO_RUN_NOFFT:
        train_model('darkir_edge_nofft', seed=seed)
else:
    print('[SKIP] RUN_NOFFT_ABLATION = False -> Bỏ qua ablation bỏ FFT/FreMLP.')


=== Bắt đầu huấn luyện darkir_edge_nofft (seed=123) [Version: v4] | 1,495,422 params | 600 steps/ep × 70 ep = 42,000 steps ===
  [Memory Step 0] Peak VRAM = 3263.8 MB | System RAM used = 9.93 GB
Epoch   1/70 | Loss: 0.2691 | LR: 2.00e-04 | Val PSNR (EMA): nan dB (Best: -1.00) | 2.2 phút
Epoch   5/70 | Loss: 0.2145 | LR: 1.98e-04 | Val PSNR (EMA): 20.60 dB (Best: 20.60) | 11.0 phút
Epoch  10/70 | Loss: 0.2032 | LR: 1.90e-04 | Val PSNR (EMA): 20.84 dB (Best: 20.84) | 22.0 phút
Epoch  15/70 | Loss: 0.1923 | LR: 1.78e-04 | Val PSNR (EMA): 21.13 dB (Best: 21.13) | 33.0 phút
Epoch  20/70 | Loss: 0.1865 | LR: 1.63e-04 | Val PSNR (EMA): 21.10 dB (Best: 21.13) | 44.0 phút
Epoch  25/70 | Loss: 0.1801 | LR: 1.44e-04 | Val PSNR (EMA): 21.20 dB (Best: 21.20) | 55.0 phút
Epoch  30/70 | Loss: 0.1751 | LR: 1.23e-04 | Val PSNR (EMA): 21.38 dB (Best: 21.38) | 65.9 phút
Epoch  35/70 | Loss: 0.1691 | LR: 1.01e-04 | Val PSNR (EMA): 21.63 dB (Best: 21.63) | 76.9 phút
Epoch  40/70 | Loss: 0.1641 | LR: 7.84e-

## 📊 Đánh giá Toàn diện trên 190 Ảnh Test (4 miền: LOLv2, Huawei, Nikon, Blur) & Kiểm định Thống kê
- **Quy tắc đăng ký trước (v3):** Đánh giá chính thức sử dụng trọng số **`final.pt` (EMA ở epoch 70)** để xuất `results_per_image.csv`, `table_main.csv` và `paired_tests.csv`. Đồng thời tự động chạy thêm một lượt đánh giá trên **`best.pt` (EMA đỉnh Val macro)** và xuất ra `results_per_image_best.csv`, `table_main_best.csv`, `paired_tests_best.csv` làm phân tích phụ.
- Tự động quét tất cả các checkpoint đã huấn luyện xong trong `CFG['ckpt_dir']` (các cell đánh giá bên dưới tự động thích ứng với số lượng seed/mô hình hiện có trong thư mục checkpoint mà không phát sinh lỗi).

In [15]:
# ===== CELL 16: Baselines cổ điển (Gamma, CLAHE, MSRCR) & Hàm tính Metrics chuẩn =====
from skimage.metrics import structural_similarity

def gamma_fn(img, g=2.2):
    return (np.power(img / 255.0, 1.0 / g) * 255.0).round().astype(np.uint8)

def clahe_fn(img, clip=2.0, tile=8):
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    l = cv2.createCLAHE(clipLimit=clip, tileGridSize=(tile, tile)).apply(l)
    return cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2RGB)

def msrcr_fn(img, sigmas=(15, 80, 250), alpha=125., beta=46., G=192., off=-30., lo_p=1, hi_p=99):
    x = img.astype(np.float64) + 1.0
    msr = sum(np.log10(x) - np.log10(cv2.GaussianBlur(x, (0, 0), s)) for s in sigmas) / len(sigmas)
    cr = beta * (np.log10(alpha * x) - np.log10(x.sum(axis=2, keepdims=True)))
    out = G * (msr * cr - off)
    for c in range(3):
        lo, hi = np.percentile(out[..., c], [lo_p, hi_p])
        out[..., c] = np.clip((out[..., c] - lo) / (hi - lo + 1e-6), 0, 1) * 255.0
    return out.round().astype(np.uint8)

LP = None
if LPIPS_OK:
    try:
        LP = lpips.LPIPS(net='alex', verbose=False).to(DEV).eval()
    except Exception as e:
        print('WARN: Không tải được trọng số AlexNet cho LPIPS -> bỏ qua LPIPS.', e)
        LPIPS_OK = False

@torch.no_grad()
def metrics(pred, gt):
    r = {
        'psnr': psnr_u8(pred, gt),
        'ssim': float(structural_similarity(pred, gt, channel_axis=2, data_range=255, gaussian_weights=True, sigma=1.5, use_sample_covariance=False))
    }
    scale = gt.mean() / max(pred.mean(), 1e-6)
    pred_norm = np.clip(pred.astype(np.float64) * scale, 0, 255).round().astype(np.uint8)
    r['psnr_gtmean'] = psnr_u8(pred_norm, gt)   # Chỉ số chẩn đoán phân rã lỗi nội bộ
    if LP is not None:
        t = lambda a: (to_tensor(a).unsqueeze(0) * 2.0 - 1.0).to(DEV)
        r['lpips'] = float(LP(t(pred), t(gt)).item())
    return r

def eval_method(name, fn, seed=None):
    rows = []
    for domain, pairs in SPLITS['test'].items():
        for lp, hp in pairs:
            lo, hi = imread_rgb(lp), imread_rgb(hp)
            rows.append({'method': name, 'seed': seed, 'domain': domain, 'image': (f'{Path(lp).parent.name}_{Path(lp).name}' if domain == 'blur' else Path(lp).name), **metrics(fn(lo), hi)})
    return pd.DataFrame(rows)

def find_checkpoint(tag, seed, ckpt_name='final.pt'):
    # v3: Mặc định nạp final.pt (EMA ở epoch 70 cố định); thứ tự fallback an toàn không trộn version
    order = (ckpt_name,) + tuple(x for x in ('final.pt', 'best.pt', 'last.pt') if x != ckpt_name)
    for name in order:
        c = Path(CFG['ckpt_dir']) / f'{tag}_seed{seed}' / name
        if c.exists():
            return c
    return None

def nn_fn(tag, seed, ckpt_name='final.pt'):
    ckpt_path = find_checkpoint(tag, seed, ckpt_name)
    if ckpt_path is None:
        raise FileNotFoundError(f"Không tìm thấy checkpoint cho {tag} (seed={seed}) trong {CFG['ckpt_dir']}.")
    ck = torch.load(ckpt_path, map_location=DEV, weights_only=False)
    m = build(tag).to(DEV).eval()
    m.load_state_dict(ck['ema'])
    return lambda img: predict_u8(m, img)

/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [00:01<00:00, 193MB/s]


In [16]:
# ===== CELL 17: Chạy đánh giá trên toàn bộ 190 ảnh test -> Lưu results_per_image.csv =====
frames = [
    eval_method('Input (Low-Light)', lambda x: x),
    eval_method('Gamma 2.2', gamma_fn),
    eval_method('CLAHE', clahe_fn),
    eval_method('MSRCR (Retinex)', msrcr_fn),
]
NAME_MAP = {'darkir_edge': 'DarkIR-Edge (ours)', 'unet': 'Standard U-Net', 'darkir_edge_nofft': 'DarkIR-Edge (no FFT)'}
ALL_CANDIDATE_RUNS = [('darkir_edge', CFG['darkir_edge_seeds']), ('unet', CFG['unet_seeds']), ('darkir_edge_nofft', CFG['darkir_edge_nofft_seeds'])]

frames_final = list(frames)
frames_best  = list(frames)

for tag, seeds in ALL_CANDIDATE_RUNS:
    for s in seeds:
        ck_fin = find_checkpoint(tag, s, ckpt_name='final.pt')
        if ck_fin is not None:
            print(f'[CHÍNH THỨC - final.pt] Đang đánh giá {NAME_MAP[tag]} (seed={s}) từ {ck_fin.name} tại: {ck_fin.parent.name}...')
            frames_final.append(eval_method(NAME_MAP[tag], nn_fn(tag, s, ckpt_name=ck_fin.name), seed=s))
        else:
            print(f'WARN: Chưa có checkpoint final.pt cho {tag} seed {s} trong {CFG["ckpt_dir"]}')

        ck_best = find_checkpoint(tag, s, ckpt_name='best.pt')
        if ck_best is not None:
            print(f'[PHÂN TÍCH PHỤ - best.pt] Đang đánh giá {NAME_MAP[tag]} (seed={s}) từ {ck_best.name} tại: {ck_best.parent.name}...')
            frames_best.append(eval_method(NAME_MAP[tag], nn_fn(tag, s, ckpt_name=ck_best.name), seed=s))

RES = pd.concat(frames_final, ignore_index=True)
res_csv_path = f"{CFG['result_dir']}/results_per_image.csv"
RES.to_csv(res_csv_path, index=False)
print(f'✔ Đã lưu chi tiết toàn bộ ảnh test (final.pt chính thức) vào: {res_csv_path}')

RES_BEST = pd.concat(frames_best, ignore_index=True)
res_best_csv_path = f"{CFG['result_dir']}/results_per_image_best.csv"
RES_BEST.to_csv(res_best_csv_path, index=False)
print(f'✔ Đã lưu chi tiết toàn bộ ảnh test (best.pt phụ) vào: {res_best_csv_path}')

# Đồng bộ bản sao vào thư mục kết quả mới nhất results_latest
if os.path.exists(CFG['latest_dir']):
    RES.to_csv(f"{CFG['latest_dir']}/results_per_image.csv", index=False)
    RES_BEST.to_csv(f"{CFG['latest_dir']}/results_per_image_best.csv", index=False)

cols_show = ['psnr', 'ssim', 'psnr_gtmean'] + (['lpips'] if LPIPS_OK and 'lpips' in RES.columns else [])
print('=== KẾT QUẢ THEO SEED (CHÍNH THỨC: final.pt EMA) ===')
print(RES.groupby(['method', 'seed'], dropna=False)[cols_show].mean().round(3))

[CHÍNH THỨC - final.pt] Đang đánh giá DarkIR-Edge (ours) (seed=42) từ final.pt tại: darkir_edge_seed42...
[PHÂN TÍCH PHỤ - best.pt] Đang đánh giá DarkIR-Edge (ours) (seed=42) từ best.pt tại: darkir_edge_seed42...
[CHÍNH THỨC - final.pt] Đang đánh giá DarkIR-Edge (ours) (seed=123) từ final.pt tại: darkir_edge_seed123...
[PHÂN TÍCH PHỤ - best.pt] Đang đánh giá DarkIR-Edge (ours) (seed=123) từ best.pt tại: darkir_edge_seed123...
[CHÍNH THỨC - final.pt] Đang đánh giá Standard U-Net (seed=42) từ final.pt tại: unet_seed42...
[PHÂN TÍCH PHỤ - best.pt] Đang đánh giá Standard U-Net (seed=42) từ best.pt tại: unet_seed42...
[CHÍNH THỨC - final.pt] Đang đánh giá Standard U-Net (seed=123) từ final.pt tại: unet_seed123...
[PHÂN TÍCH PHỤ - best.pt] Đang đánh giá Standard U-Net (seed=123) từ best.pt tại: unet_seed123...
[CHÍNH THỨC - final.pt] Đang đánh giá DarkIR-Edge (no FFT) (seed=42) từ final.pt tại: darkir_edge_nofft_seed42...
[PHÂN TÍCH PHỤ - best.pt] Đang đánh giá DarkIR-Edge (no FFT) (seed=42)

In [17]:
# ===== CELL 18: Tổng hợp Bảng kết quả chính (Mean ± Std qua seed) & Kiểm định ghép cặp (Paired Tests) =====
from scipy import stats
METRICS = [m for m in ['psnr', 'ssim', 'lpips'] if m in RES.columns]

def summarize(df):
    rows = []
    for (meth, seed), g in df.groupby(['method', 'seed'], dropna=False):
        r = {'method': meth, 'seed': seed}
        for m in METRICS:
            dom = g.groupby('domain')[m].mean()
            for d, v in dom.items(): r[f'{m}|{d}'] = v
            r[f'{m}|pooled'] = g[m].mean()
            r[f'{m}|macro']  = dom.mean()
        rows.append(r)
    per_seed = pd.DataFrame(rows)
    agg = per_seed.drop(columns='seed').groupby('method', sort=False).agg(['mean', 'std'])
    return per_seed, agg

per_seed, agg = summarize(RES)
def fmt(meth, col):
    mu, sd = agg.loc[meth, (col, 'mean')], agg.loc[meth, (col, 'std')]
    return f'{mu:.3f}' + (f' ± {sd:.3f}' if not np.isnan(sd) else '')

cols = [c for c in per_seed.columns if c not in ('method', 'seed')]
TABLE = pd.DataFrame({c: [fmt(m, c) for m in agg.index] for c in cols}, index=agg.index)

# Lưu bảng kết quả chính vào folder version trên Drive
table_path = f"{CFG['result_dir']}/table_main.csv"
TABLE.to_csv(table_path)
print(f'✔ Đã lưu Bảng kết quả chính vào: {table_path}')

# Đồng bộ bản sao sang results_latest
if os.path.exists(CFG['latest_dir']):
    TABLE.to_csv(f"{CFG['latest_dir']}/table_main.csv")

pd.set_option('display.width', 260, 'display.max_columns', 50)
print('=== BẢNG PSNR (dB, cao hơn là tốt hơn) ===')
print(TABLE[[c for c in cols if c.startswith('psnr|')]])
print()
print('=== BẢNG SSIM (cao hơn là tốt hơn) ===')
print(TABLE[[c for c in cols if c.startswith('ssim|')]])
if LPIPS_OK and any(c.startswith('lpips|') for c in cols):
    print()
    print('=== BẢNG LPIPS (thấp hơn là tốt hơn) ===')
    print(TABLE[[c for c in cols if c.startswith('lpips|')]])

# --- Bảng chẩn đoán phân rã lỗi GT-mean (Diagnostic Table) ---
diag = RES.groupby(['method', 'domain'])[['psnr', 'psnr_gtmean']].mean()
diag['gap_illumination_dB'] = diag['psnr_gtmean'] - diag['psnr']
diag_path = f"{CFG['result_dir']}/diagnostic_gtmean.csv"
diag.to_csv(diag_path)
if os.path.exists(CFG['latest_dir']):
    diag.to_csv(f"{CFG['latest_dir']}/diagnostic_gtmean.csv")
print()
print('=== CHẨN ĐOÁN NỘI BỘ: ĐỘ LỆCH ÁNH SÁNG TOÀN CỤC (psnr_gtmean - psnr) ===')
print(diag.round(2))

# --- Kiểm định thống kê ghép cặp từng ảnh (Paired t-test, Wilcoxon & Bootstrap 95% CI) ---
# ΔPSNR = PSNR(A) − PSNR(B) trên từng ảnh test (trung bình qua các seed); ΔPSNR > 0 nghĩa là A tốt hơn B.
A = 'DarkIR-Edge (ours)'
B = 'Standard U-Net' if 'Standard U-Net' in RES.method.values else 'CLAHE'
COMPARISONS = [(A, B)] + ([(A, 'DarkIR-Edge (no FFT)')] if 'DarkIR-Edge (no FFT)' in RES.method.values else [])

def paired_report(a, b, domain=None):
    sub = RES if domain is None else RES[RES.domain == domain]
    pa = sub[sub.method == a].groupby(['domain', 'image']).psnr.mean()
    pb = sub[sub.method == b].groupby(['domain', 'image']).psnr.mean()
    idx = pa.index.intersection(pb.index)
    d = (pa.loc[idx] - pb.loc[idx]).values
    if len(d) < 2:
        return None
    rng = np.random.default_rng(MASTER_SEED)
    ci = np.percentile([rng.choice(d, len(d)).mean() for _ in range(2000)], [2.5, 97.5])
    t_pval = stats.ttest_rel(pa.loc[idx], pb.loc[idx]).pvalue
    w_pval = stats.wilcoxon(d).pvalue if np.any(d != 0) else 1.0
    return dict(a=a, b=b, domain=domain or 'pooled', n=len(d), mean_delta_psnr=d.mean(), ci_lo=ci[0], ci_hi=ci[1],
                t_p=t_pval, wilcoxon_p=w_pval, frac_a_better=(d > 0).mean())

PAIRED = pd.DataFrame([r for a_, b_ in COMPARISONS for dom in [None] + sorted(RES.domain.unique())
                       if (r := paired_report(a_, b_, dom)) is not None])
paired_path = f"{CFG['result_dir']}/paired_tests.csv"
PAIRED.to_csv(paired_path, index=False)
if os.path.exists(CFG['latest_dir']):
    PAIRED.to_csv(f"{CFG['latest_dir']}/paired_tests.csv", index=False)
print()
print('=== KIỂM ĐỊNH GHÉP CẶP ΔPSNR (A − B), theo từng miền và gộp ===')
print(PAIRED.round(4).to_string(index=False))
print(f'✔ Đã lưu bảng kiểm định ghép cặp vào: {paired_path}')

# --- Xuất thêm Bảng phân tích phụ từ best.pt (table_main_best.csv & paired_tests_best.csv) ---
if 'RES_BEST' in globals() and len(RES_BEST) > 0:
    ps_b, agg_b = summarize(RES_BEST)
    def fmt_b(meth, col):
        mu, sd = agg_b.loc[meth, (col, 'mean')], agg_b.loc[meth, (col, 'std')]
        return f'{mu:.3f}' + (f' ± {sd:.3f}' if not np.isnan(sd) else '')
    cols_b = [c for c in ps_b.columns if c not in ('method', 'seed')]
    TABLE_BEST = pd.DataFrame({c: [fmt_b(m, c) for m in agg_b.index] for c in cols_b}, index=agg_b.index)
    table_best_path = f"{CFG['result_dir']}/table_main_best.csv"
    TABLE_BEST.to_csv(table_best_path)
    if os.path.exists(CFG['latest_dir']):
        TABLE_BEST.to_csv(f"{CFG['latest_dir']}/table_main_best.csv")
    print()
    print('=== [PHÂN TÍCH PHỤ - best.pt EMA] BẢNG PSNR (dB) ===')
    print(TABLE_BEST[[c for c in cols_b if c.startswith('psnr|')]])
    print(f'✔ Đã lưu Bảng kết quả phụ (best.pt) vào: {table_best_path}')

# Liệt kê tất cả các phiên bản thực nghiệm đã có trong Google Drive:
exp_base = Path(DRIVE) / 'experiments'
if exp_base.exists():
    all_runs = sorted([d.name for d in exp_base.iterdir() if d.is_dir()])
    print()
    print(f"📁 Danh sách các phiên bản thực nghiệm đã lưu trong Google Drive ({len(all_runs)} runs): {all_runs}")

✔ Đã lưu Bảng kết quả chính vào: /content/drive/MyDrive/llie_project_v4/experiments/v4/results/table_main.csv
=== BẢNG PSNR (dB, cao hơn là tốt hơn) ===
                           psnr|blur     psnr|huawei      psnr|lolv2      psnr|nikon     psnr|pooled      psnr|macro
method                                                                                                              
CLAHE                         12.899          10.392          11.537          11.040          11.590          11.467
DarkIR-Edge (no FFT)  23.717 ± 0.256  20.658 ± 0.027  20.975 ± 0.489  17.132 ± 0.006  21.098 ± 0.316  20.620 ± 0.195
DarkIR-Edge (ours)    23.537 ± 0.009  20.968 ± 0.195  20.740 ± 0.019  16.961 ± 0.166  20.967 ± 0.060  20.552 ± 0.098
Gamma 2.2                     18.092          15.078          17.268          15.416          16.901          16.463
Input (Low-Light)             10.549           9.062           9.718           9.303           9.746           9.658
MSRCR (Retinex)             

In [18]:
# ===== CELL 19: Xuất lưới ảnh định tính (Qualitative Grid), Đường cong học tập & Ảnh không cặp (DICM/LIME) =====
import matplotlib.pyplot as plt

def pick_examples(k=2):
    out = []
    for d, pairs in SPLITS['test'].items():
        out += [(d, p) for p in pairs[:k]]
    return out

ex = pick_examples(2)
fns = {'Input': lambda x: x, 'CLAHE': clahe_fn}
for s in CFG['unet_seeds']:
    ckpt_file = find_checkpoint('unet', s)
    if ckpt_file is not None:
        fns['Standard U-Net'] = nn_fn('unet', s, ckpt_name=ckpt_file.name)
        break

for s in CFG['darkir_edge_nofft_seeds']:
    ckpt_file = find_checkpoint('darkir_edge_nofft', s)
    if ckpt_file is not None:
        fns['DarkIR-Edge (no FFT)'] = nn_fn('darkir_edge_nofft', s, ckpt_name=ckpt_file.name)
        break

for s in CFG['darkir_edge_seeds']:
    ckpt_file = find_checkpoint('darkir_edge', s)
    if ckpt_file is not None:
        fns['DarkIR-Edge'] = nn_fn('darkir_edge', s, ckpt_name=ckpt_file.name)
        break

fig, ax = plt.subplots(len(ex), len(fns) + 1, figsize=(3.2 * (len(fns) + 1), 2.5 * len(ex)))
ax = np.atleast_2d(ax)
for r, (dom, (lp, hp)) in enumerate(ex):
    lo, hi = imread_rgb(lp), imread_rgb(hp)
    for c, (name, fn) in enumerate(fns.items()):
        im = fn(lo)
        ax[r, c].imshow(im)
        title = f'{name} ({psnr_u8(im, hi):.2f} dB)' if name != 'Input' else f'Input ({dom}, {psnr_u8(lo, hi):.2f} dB)'
        ax[r, c].set_title(title, fontsize=8.5)
        ax[r, c].axis('off')
    ax[r, -1].imshow(hi)
    ax[r, -1].set_title('Ground Truth', fontsize=8.5)
    ax[r, -1].axis('off')
plt.tight_layout()
qual_path = f"{CFG['result_dir']}/qualitative.png"
plt.savefig(qual_path, dpi=150)
if os.path.exists(CFG['latest_dir']):
    plt.savefig(f"{CFG['latest_dir']}/qualitative.png", dpi=150)
plt.show()
print(f'✔ Đã lưu Lưới ảnh so sánh định tính vào: {qual_path}')

# --- Tự động lưu từng ảnh phục chế mẫu kích thước gốc vào folder visual_samples trên Drive ---
vis_dir = Path(CFG['result_dir']) / 'visual_samples'
vis_dir.mkdir(parents=True, exist_ok=True)
for r, (dom, (lp, hp)) in enumerate(ex):
    base_name = Path(lp).stem
    cv2.imwrite(str(vis_dir / f'{r+1:02d}_{dom}_{base_name}_input.png'), cv2.cvtColor(imread_rgb(lp), cv2.COLOR_RGB2BGR))
    cv2.imwrite(str(vis_dir / f'{r+1:02d}_{dom}_{base_name}_gt.png'), cv2.cvtColor(imread_rgb(hp), cv2.COLOR_RGB2BGR))
    for name, fn in fns.items():
        if name != 'Input':
            san_name = name.lower().replace(' ', '_').replace('-', '_').replace('(', '').replace(')', '')
            cv2.imwrite(str(vis_dir / f'{r+1:02d}_{dom}_{base_name}_{san_name}.png'), cv2.cvtColor(fn(imread_rgb(lp)), cv2.COLOR_RGB2BGR))
print(f'✔ Đã lưu các ảnh phục chế mẫu độ phân giải cao vào: {vis_dir}')

# --- Vẽ đường cong huấn luyện (Loss & Validation PSNR) ---
fig, ax = plt.subplots(1, 2, figsize=(10.5, 3.5))
has_history = False
for tag, seeds in [('darkir_edge', CFG['darkir_edge_seeds']), ('unet', CFG['unet_seeds']), ('darkir_edge_nofft', CFG['darkir_edge_nofft_seeds'])]:
    for s in seeds:
        h = Path(CFG['ckpt_dir']) / f'{tag}_seed{s}' / 'history.csv'
        if h.exists():
            has_history = True
            df = pd.read_csv(h)
            ax[0].plot(df.epoch, df.loss, label=f'{tag} (s={s})')
            v = df.dropna(subset=['val_psnr_ema'])
            ax[1].plot(v.epoch, v.val_psnr_ema, marker='o', markersize=3, label=f'{tag} (s={s})')

if has_history:
    ax[0].set_title('Training Composite Loss'); ax[0].set_xlabel('Epoch'); ax[0].legend(); ax[0].grid(alpha=0.3)
    ax[1].set_title('Validation PSNR macro (EMA, dB)'); ax[1].set_xlabel('Epoch'); ax[1].legend(); ax[1].grid(alpha=0.3)
    plt.tight_layout()
    curve_path = f"{CFG['result_dir']}/curves.png"
    plt.savefig(curve_path, dpi=150)
    if os.path.exists(CFG['latest_dir']):
        plt.savefig(f"{CFG['latest_dir']}/curves.png", dpi=150)
    plt.show()
    print(f'✔ Đã lưu Biểu đồ đường cong học tập vào: {curve_path}')
else:
    plt.close()

Output hidden; open in https://colab.research.google.com to view.

In [19]:
# ===== CELL 20: Bảng Hiệu năng Phần cứng (Params, GFLOPs, Latency FP32/FP16, FPS, VRAM) =====
from torch.utils.flop_counter import FlopCounterMode

rows = []
# Lưu ý khi báo cáo: FLOPs chưa tính phần FFT; latency/VRAM chỉ đúng với GPU ghi ở cột GPU (không trộn số đo T4 với L4).
for tag, name in [('darkir_edge', 'DarkIR-Edge (C=24)'), ('darkir_edge_nofft', 'DarkIR-Edge no-FFT (C=24)'), ('unet', 'Standard U-Net (base=32)')]:
    m = build(tag).to(DEV).eval()
    r = {'Model': name, 'Params (M)': round(n_params(m) / 1e6, 3), 'Exact Params': n_params(m),
         'GPU': torch.cuda.get_device_name(0) if DEV.type == 'cuda' else 'cpu'}
    with FlopCounterMode(display=False) as fc, torch.no_grad():
        m(torch.rand(1, 3, 256, 256, device=DEV))
    r['GFLOPs @ 256x256 (Spatial)'] = round(fc.get_total_flops() / 1e9, 2)
    if DEV.type == 'cuda':
        x_bench = torch.rand(1, 3, 400, 600, device=DEV)
        for prec, dt in [('FP32', None), ('FP16', torch.float16)]:
            with torch.no_grad(), torch.autocast('cuda', dtype=torch.float16, enabled=dt is not None):
                for _ in range(10): m(x_bench)
                torch.cuda.synchronize()
                torch.cuda.reset_peak_memory_stats()
                t0 = time.time()
                for _ in range(50): m(x_bench)
                torch.cuda.synchronize()
                ms = (time.time() - t0) / 50.0 * 1000.0
            r[f'Latency {prec} @600x400 (ms)'] = round(ms, 2)
            r[f'FPS {prec} @600x400'] = round(1000.0 / ms, 1)
        r['Peak VRAM (MB)'] = round(torch.cuda.max_memory_allocated() / (1024 ** 2), 1)
    rows.append(r)

EFF = pd.DataFrame(rows)
eff_path = f"{CFG['result_dir']}/efficiency.csv"
EFF.to_csv(eff_path, index=False)
if os.path.exists(CFG['latest_dir']):
    EFF.to_csv(f"{CFG['latest_dir']}/efficiency.csv", index=False)
print(f'✔ Đã lưu Bảng hiệu năng phần cứng vào: {eff_path}')
print(EFF.T)

✔ Đã lưu Bảng hiệu năng phần cứng vào: /content/drive/MyDrive/llie_project_v4/experiments/v4/results/efficiency.csv
                                             0                          1                         2
Model                       DarkIR-Edge (C=24)  DarkIR-Edge no-FFT (C=24)  Standard U-Net (base=32)
Params (M)                               1.924                      1.495                     1.928
Exact Params                           1924038                    1495422                   1928483
GPU                                  NVIDIA L4                  NVIDIA L4                 NVIDIA L4
GFLOPs @ 256x256 (Spatial)                8.75                        7.5                     18.45
Latency FP32 @600x400 (ms)               32.29                      27.79                     13.37
FPS FP32 @600x400                         31.0                       36.0                      74.8
Latency FP16 @600x400 (ms)               26.63                      21.98           

## (Tùy chọn) Đánh giá Mốc Trần Tham Chiếu `DarkIR-m` Chính Thức (CVPR 2025) trên 150 Ảnh Test
Vì lớp `DarkIREdge(width=32, use_imax=False, use_gate=False)` ở Cell 8 khớp 1:1 tên biến và cấu trúc của `DarkIR/archs/DarkIR.py`, cờ `RUN_OFFICIAL_DARKIR_REF = True` ở Cell 22 dưới đây cho phép tự động tải trọng số công bố từ HuggingFace (`Cidaut/DarkIR`) và đánh giá làm mốc trần tham chiếu.

In [20]:
# ===== CELL 22 (Tùy chọn): Tự động tải & chạy Checkpoint DarkIR-m (C=32) chính thức từ HuggingFace =====
RUN_OFFICIAL_DARKIR_REF = not TEST_MODE   # Đặt True để thực thi đánh giá mốc trần tham chiếu DarkIR-m (CVPR 2025)

if RUN_OFFICIAL_DARKIR_REF:
    try:
        from huggingface_hub import hf_hub_download, list_repo_files
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub'])
        from huggingface_hub import hf_hub_download, list_repo_files

    files = [f for f in list_repo_files('Cidaut/DarkIR') if f.endswith('.pt') or f.endswith('.pth')]
    print('Các checkpoint có trong HuggingFace Cidaut/DarkIR:', files)
    target_ckpt = next((f for f in files if '1k' in f or '384' in f or 'lol' in f.lower()), files[0] if files else None)
    if target_ckpt:
        ckpt_path = hf_hub_download(repo_id='Cidaut/DarkIR', filename=target_ckpt)
        raw = torch.load(ckpt_path, map_location=DEV, weights_only=False)
        state = raw.get('params', raw.get('model_state_dict', raw))
        clean_state = {}
        for k, v in state.items():
            k2 = k.replace('module.', '').replace('.modules_list.', '.')
            if 'adapter' not in k2:
                clean_state[k2] = v
        ref_model = DarkIREdge(width=32, use_imax=False, use_gate=False).to(DEV).eval()
        missing, unexpected = ref_model.load_state_dict(clean_state, strict=False)
        print(f'Đã nạp {target_ckpt} | missing={len(missing)}, unexpected={len(unexpected)}')
        ref_df = eval_method('DarkIR-m (Official CVPR25 Ref)', lambda im: predict_u8(ref_model, im), seed=0)
        RES_WITH_REF = pd.concat([RES, ref_df], ignore_index=True)
        ref_path = f"{CFG['result_dir']}/results_per_image_with_ref.csv"
        RES_WITH_REF.to_csv(ref_path, index=False)
        if os.path.exists(CFG['latest_dir']):
            RES_WITH_REF.to_csv(f"{CFG['latest_dir']}/results_per_image_with_ref.csv", index=False)
        print(f'✔ Đã lưu kết quả mốc trần tham chiếu vào: {ref_path}')
        print(RES_WITH_REF.groupby('method')[['psnr', 'ssim']].mean().round(3))


Các checkpoint có trong HuggingFace Cidaut/DarkIR: ['DarkIR_384.pt']


DarkIR_384.pt: reconstructing file:   0%|          |  0.00B / 13.4MB            

DarkIR_384.pt: downloading bytes:           |  0.00B            

Đã nạp DarkIR_384.pt | missing=0, unexpected=0
✔ Đã lưu kết quả mốc trần tham chiếu vào: /content/drive/MyDrive/llie_project_v4/experiments/v4/results/results_per_image_with_ref.csv
                                  psnr   ssim
method                                       
CLAHE                           11.590  0.405
DarkIR-Edge (no FFT)            21.098  0.743
DarkIR-Edge (ours)              20.967  0.743
DarkIR-m (Official CVPR25 Ref)  20.963  0.728
Gamma 2.2                       16.901  0.595
Input (Low-Light)                9.746  0.217
MSRCR (Retinex)                 12.122  0.410
Standard U-Net                  21.030  0.745


In [21]:
# ===== CELL 23: Benchmark trực quan với DarkIR-m (CVPR'25) trên Test Set & Tải 5 ảnh LOL-Blur từ HuggingFace =====
import urllib.request
import matplotlib.pyplot as plt

# 1. Tự động nạp DarkIR-m (CVPR'25) từ HuggingFace nếu chưa có trong bộ nhớ
if 'ref_model' not in globals():
    try:
        from huggingface_hub import hf_hub_download, list_repo_files
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub'])
        from huggingface_hub import hf_hub_download, list_repo_files
    files = [f for f in list_repo_files('Cidaut/DarkIR') if f.endswith('.pt') or f.endswith('.pth')]
    target_ckpt = next((f for f in files if '1k' in f or '384' in f or 'lol' in f.lower()), files[0] if files else 'DarkIR_384.pt')
    ckpt_path = hf_hub_download(repo_id='Cidaut/DarkIR', filename=target_ckpt)
    raw = torch.load(ckpt_path, map_location=DEV, weights_only=False)
    state = raw.get('params', raw.get('model_state_dict', raw))
    clean_state = {k.replace('module.', '').replace('.modules_list.', '.'): v for k, v in state.items() if 'adapter' not in k}
    ref_model = DarkIREdge(width=32, use_imax=False, use_gate=False).to(DEV).eval()
    ref_model.load_state_dict(clean_state, strict=False)
    print(f'✔ Đã nạp thành công {target_ckpt} (DarkIR-m CVPR 2025) từ HuggingFace Cidaut/DarkIR')

# 2. Khởi tạo danh sách hàm dự đoán để đối sánh trực quan
bench_fns = {'Input': lambda x: x, 'CLAHE': clahe_fn}
for s in CFG['unet_seeds']:
    ck_u = find_checkpoint('unet', s)
    if ck_u is not None:
        bench_fns['Standard U-Net'] = nn_fn('unet', s, ckpt_name=ck_u.name); break
bench_fns['DarkIR-m (CVPR25)'] = lambda im: predict_u8(ref_model, im)
for s in CFG['darkir_edge_nofft_seeds']:
    ck_nf = find_checkpoint('darkir_edge_nofft', s)
    if ck_nf is not None:
        bench_fns['DarkIR-Edge (no FFT)'] = nn_fn('darkir_edge_nofft', s, ckpt_name=ck_nf.name); break
for s in CFG['darkir_edge_seeds']:
    ck_d = find_checkpoint('darkir_edge', s)
    if ck_d is not None:
        bench_fns['DarkIR-Edge (Ours)'] = nn_fn('darkir_edge', s, ckpt_name=ck_d.name); break

# 3. PHẦN A: Benchmark trực quan 1 ảnh mẫu/domain trên tập Test (có Ground Truth & PSNR/SSIM)
ex_pairs = [(d, p) for d, pairs in SPLITS['test'].items() for p in pairs[:1]]
fig, ax = plt.subplots(len(ex_pairs), len(bench_fns) + 1, figsize=(3.2 * (len(bench_fns) + 1), 2.6 * len(ex_pairs)))
ax = np.atleast_2d(ax)
for r, (dom, (lp, hp)) in enumerate(ex_pairs):
    lo, hi = imread_rgb(lp), imread_rgb(hp)
    for c, (name, fn) in enumerate(bench_fns.items()):
        im = fn(lo)
        ax[r, c].imshow(im)
        p_val = psnr_u8(im, hi)
        s_val = structural_similarity(im, hi, channel_axis=2, data_range=255, gaussian_weights=True, sigma=1.5, use_sample_covariance=False)
        title = f'{name}\n{p_val:.2f} dB | SSIM {s_val:.3f}' if name != 'Input' else f'Input ({dom})\n{p_val:.2f} dB'
        ax[r, c].set_title(title, fontsize=8)
        ax[r, c].axis('off')
    ax[r, -1].imshow(hi)
    ax[r, -1].set_title('Ground Truth', fontsize=8)
    ax[r, -1].axis('off')
plt.tight_layout()
q_ref_path = f"{CFG['result_dir']}/qualitative_with_darkir_m.png"
plt.savefig(q_ref_path, dpi=150)
if os.path.exists(CFG['latest_dir']):
    plt.savefig(f"{CFG['latest_dir']}/qualitative_with_darkir_m.png", dpi=150)
plt.show()
print(f'✔ Đã lưu lưới ảnh so sánh có DarkIR-m (CVPR 2025) tại: {q_ref_path}')

# 4. PHẦN B: Tải 5 ảnh LOL-Blur từ HuggingFace Space (Cidaut/DarkIR) & Benchmark trực quan
lolblur_dir = Path(CFG['data_root']) / 'LOLBlur_hf_samples'
lolblur_dir.mkdir(parents=True, exist_ok=True)
lolblur_out_dir = Path(CFG['result_dir']) / 'lolblur_visual_samples'
lolblur_out_dir.mkdir(parents=True, exist_ok=True)

hf_lolblur_urls = {
    '0010.png': 'https://huggingface.co/spaces/Cidaut/DarkIR/resolve/main/examples/0010.png',
    '0087.png': 'https://huggingface.co/spaces/Cidaut/DarkIR/resolve/main/examples/0087.png',
    '1001.png': 'https://huggingface.co/spaces/Cidaut/DarkIR/resolve/main/examples/1001.png',
    '1100.png': 'https://huggingface.co/spaces/Cidaut/DarkIR/resolve/main/examples/1100.png',
    '0075.png': 'https://raw.githubusercontent.com/sczhou/LEDNet/master/inputs/synthetic/0075.png',
}

lb_paths = []
for fname, url in hf_lolblur_urls.items():
    dst = lolblur_dir / fname
    if not dst.exists():
        urllib.request.urlretrieve(url, dst)
    lb_paths.append(dst)
print(f'✔ Đã tải {len(lb_paths)} ảnh mẫu LOL-Blur vào: {lolblur_dir}')

fig2, ax2 = plt.subplots(len(lb_paths), len(bench_fns), figsize=(3.2 * len(bench_fns), 2.5 * len(lb_paths)))
ax2 = np.atleast_2d(ax2)
for r, p in enumerate(lb_paths):
    lo = imread_rgb(str(p))
    for c, (name, fn) in enumerate(bench_fns.items()):
        im = fn(lo)
        ax2[r, c].imshow(im)
        ax2[r, c].set_title(f'{name} ({p.name})' if c == 0 else name, fontsize=8.5)
        ax2[r, c].axis('off')
        san = name.lower().replace(' ', '_').replace('-', '_').replace('(', '').replace(')', '')
        cv2.imwrite(str(lolblur_out_dir / f'{p.stem}_{san}.png'), cv2.cvtColor(im, cv2.COLOR_RGB2BGR))
plt.tight_layout()
lb_grid_path = f"{CFG['result_dir']}/qualitative_lolblur_5samples.png"
plt.savefig(lb_grid_path, dpi=150)
if os.path.exists(CFG['latest_dir']):
    plt.savefig(f"{CFG['latest_dir']}/qualitative_lolblur_5samples.png", dpi=150)
plt.show()
print(f'✔ Đã lưu lưới ảnh benchmark 5 mẫu LOL-Blur tại: {lb_grid_path}')
print(f'✔ Đã lưu từng ảnh LOL-Blur độ phân giải gốc tại: {lolblur_out_dir}')

Output hidden; open in https://colab.research.google.com to view.

## 🔎 Đánh giá Định lượng Bổ sung (Mục 4 `ke_hoach_v3.md` — Kiểm chứng Độ sáng, Phân bố Tông, Màu sắc CIEDE2000 & Độ nét Laplacian)
Cell dưới đây định lượng hóa trực tiếp các nhận xét thị giác trên toàn bộ 190 ảnh test mà không cần huấn luyện lại:
1. **Sai lệch độ sáng có dấu (`lum_bias`):** $\mu_Y(\text{out}) - \mu_Y(\text{GT})$ (trên thang $[0, 255]$; dương = quá sáng, âm = thiếu sáng) và sai số tuyệt đối `lum_abs_err` $|\mu_Y(\text{out}) - \mu_Y(\text{GT})|$.
2. **Khoảng cách histogram độ chói (`lum_emd`):** Wasserstein-1 (Earth Mover's Distance 1D) giữa phân bố độ chói $Y(\text{out})$ và $Y(\text{GT})$.
3. **Sai lệch màu sắc cảm nhận (`ciede2000` / $\Delta E_{00}$):** Khoảng cách màu chuẩn CIE 2000 trên không gian LAB (thấp hơn là màu sắc chân thực hơn).
4. **Tỉ lệ độ nét Laplacian (`sharp_ratio`):** $\text{Var}(\nabla^2 Y_{\text{out}}) / \text{Var}(\nabla^2 Y_{\text{GT}})$ (kiểm chứng trực tiếp mức độ khử mờ trên miền `blur`).
5. **Sai lệch trên 10% vùng tối nhất & 10% vùng sáng nhất của GT (`dark10_mae`, `bright10_mae`):** Kiểm tra giả thuyết *"vùng cần tối thì tối, vùng cần sáng thì sáng"*.


In [22]:
# ===== CELL 24 (MỚI - v3): Đo lường bổ sung (Luminance Bias, EMD Histogram, CIEDE2000, Laplacian Sharpness, Dark/Bright 10%) =====
from skimage.color import rgb2lab, deltaE_ciede2000
from scipy.stats import wasserstein_distance

def extra_image_metrics(pred_u8, gt_u8, inp_u8):
    # Độ chói chuẩn ITU-R BT.601 trên thang [0, 255]
    y_out = 0.299 * pred_u8[..., 0].astype(np.float64) + 0.587 * pred_u8[..., 1].astype(np.float64) + 0.114 * pred_u8[..., 2].astype(np.float64)
    y_gt  = 0.299 * gt_u8[..., 0].astype(np.float64)   + 0.587 * gt_u8[..., 1].astype(np.float64)   + 0.114 * gt_u8[..., 2].astype(np.float64)
    y_in  = 0.299 * inp_u8[..., 0].astype(np.float64)  + 0.587 * inp_u8[..., 1].astype(np.float64)  + 0.114 * inp_u8[..., 2].astype(np.float64)

    lum_bias = float(y_out.mean() - y_gt.mean())
    lum_abs_err = float(abs(lum_bias))
    lum_emd = float(wasserstein_distance(y_out.ravel(), y_gt.ravel()))

    # Sai lệch màu CIEDE2000 (ΔE00)
    lab_out = rgb2lab(pred_u8.astype(np.float32) / 255.0)
    lab_gt  = rgb2lab(gt_u8.astype(np.float32) / 255.0)
    de00 = float(deltaE_ciede2000(lab_out, lab_gt).mean())

    # Phương sai Laplacian (độ nét)
    lap_out = float(cv2.Laplacian(y_out, cv2.CV_64F).var())
    lap_gt  = float(cv2.Laplacian(y_gt,  cv2.CV_64F).var())
    lap_in  = float(cv2.Laplacian(y_in,  cv2.CV_64F).var())
    sharp_ratio = float(lap_out / max(lap_gt, 1e-6))

    # Sai số trên 10% pixel tối nhất và 10% pixel sáng nhất của GT
    p10, p90 = np.percentile(y_gt, [10, 90])
    dark_mask   = y_gt <= p10
    bright_mask = y_gt >= p90
    dark10_bias   = float((y_out[dark_mask] - y_gt[dark_mask]).mean())
    dark10_mae    = float(np.abs(y_out[dark_mask] - y_gt[dark_mask]).mean())
    bright10_bias = float((y_out[bright_mask] - y_gt[bright_mask]).mean())
    bright10_mae  = float(np.abs(y_out[bright_mask] - y_gt[bright_mask]).mean())

    return dict(
        lum_bias=lum_bias, lum_abs_err=lum_abs_err, lum_emd=lum_emd, ciede2000=de00,
        lap_in=lap_in, lap_out=lap_out, lap_gt=lap_gt, sharp_ratio=sharp_ratio,
        dark10_bias=dark10_bias, dark10_mae=dark10_mae,
        bright10_bias=bright10_bias, bright10_mae=bright10_mae
    )

extra_rows = []
for tag, seeds in ALL_CANDIDATE_RUNS:
    for s in seeds:
        ck = find_checkpoint(tag, s, ckpt_name='final.pt')
        if ck is None:
            continue
        fn = nn_fn(tag, s, ckpt_name=ck.name)
        for domain, pairs in SPLITS['test'].items():
            for lp, hp in pairs:
                lo, hi = imread_rgb(lp), imread_rgb(hp)
                pred = fn(lo)
                img_id = f'{Path(lp).parent.name}_{Path(lp).name}' if domain == 'blur' else Path(lp).name
                extra_rows.append({'method': NAME_MAP[tag], 'seed': s, 'domain': domain, 'image': img_id, **extra_image_metrics(pred, hi, lo)})

if extra_rows:
    RES_EXTRA = pd.DataFrame(extra_rows)
    extra_csv = f"{CFG['result_dir']}/results_extra.csv"
    RES_EXTRA.to_csv(extra_csv, index=False)
    if os.path.exists(CFG['latest_dir']):
        RES_EXTRA.to_csv(f"{CFG['latest_dir']}/results_extra.csv", index=False)

    extra_cols = ['lum_bias', 'lum_abs_err', 'lum_emd', 'ciede2000', 'sharp_ratio', 'dark10_mae', 'bright10_mae']
    table_extra = RES_EXTRA.groupby(['method', 'domain'])[extra_cols].mean().round(3)
    table_extra_path = f"{CFG['result_dir']}/table_extra.csv"
    table_extra.to_csv(table_extra_path)
    if os.path.exists(CFG['latest_dir']):
        table_extra.to_csv(f"{CFG['latest_dir']}/table_extra.csv")
    print('=== BẢNG CHỈ SỐ BỔ SUNG THEO MIỀN (Trung bình qua các seed) ===')
    print(table_extra)

    # Kiểm định ghép cặp bootstrap cho các chỉ số bổ sung
    def paired_extra_metric(df, a, b, col, domain=None):
        sub = df if domain is None else df[df.domain == domain]
        va = sub[sub.method == a].groupby(['domain', 'image'])[col].mean()
        vb = sub[sub.method == b].groupby(['domain', 'image'])[col].mean()
        idx = va.index.intersection(vb.index)
        d = (va.loc[idx] - vb.loc[idx]).values
        if len(d) < 2:
            return None
        rng = np.random.default_rng(MASTER_SEED)
        ci = np.percentile([rng.choice(d, len(d)).mean() for _ in range(2000)], [2.5, 97.5])
        t_pval = stats.ttest_rel(va.loc[idx], vb.loc[idx]).pvalue
        w_pval = stats.wilcoxon(d).pvalue if np.any(d != 0) else 1.0
        return dict(metric=col, a=a, b=b, domain=domain or 'pooled', n=len(d),
                    mean_a=va.loc[idx].mean(), mean_b=vb.loc[idx].mean(),
                    mean_delta=d.mean(), ci_lo=ci[0], ci_hi=ci[1], t_p=t_pval, wilcoxon_p=w_pval)

    extra_paired = pd.DataFrame([
        r for col in ['lum_abs_err', 'lum_emd', 'ciede2000', 'sharp_ratio', 'dark10_mae', 'bright10_mae']
        for a_, b_ in COMPARISONS
        for dom in [None] + sorted(RES_EXTRA.domain.unique())
        if (r := paired_extra_metric(RES_EXTRA, a_, b_, col, dom)) is not None
    ])
    extra_paired_path = f"{CFG['result_dir']}/paired_tests_extra.csv"
    extra_paired.to_csv(extra_paired_path, index=False)
    if os.path.exists(CFG['latest_dir']):
        extra_paired.to_csv(f"{CFG['latest_dir']}/paired_tests_extra.csv", index=False)
    print()
    print('=== KIỂM ĐỊNH GHÉP CẶP CÁC CHỈ SỐ BỔ SUNG (POOLED & BLUR) ===')
    print(extra_paired[extra_paired.domain.isin(['pooled', 'blur'])].round(4).to_string(index=False))
    print(f'✔ Đã lưu kết quả đo bổ sung tại: {extra_csv} và {extra_paired_path}')


=== BẢNG CHỈ SỐ BỔ SUNG THEO MIỀN (Trung bình qua các seed) ===
                             lum_bias  lum_abs_err  lum_emd  ciede2000  sharp_ratio  dark10_mae  bright10_mae
method               domain                                                                                  
DarkIR-Edge (no FFT) blur       0.665       10.612   10.928      6.181        0.192      11.824        13.596
                     huawei    -6.199       13.391   14.431      8.458        0.085      10.854        24.930
                     lolv2     16.800       20.645   21.472     10.661        0.288      13.474        28.205
                     nikon      1.240       19.833   20.551     10.706        0.204      22.508        23.693
DarkIR-Edge (ours)   blur      -0.410       10.634   10.956      6.124        0.192      11.343        13.484
                     huawei    -8.225       12.764   13.663      8.424        0.085      10.460        24.232
                     lolv2     14.954       20.328   21.

## 🔄 Kiểm định Đối chứng Ghép cặp `v3 ↔ v4` (`paired_tests_v3_v4.csv`) & Đánh giá Không Tham Chiếu `Real-LOLBlur` (482 ảnh)
Thực hiện đúng quy tắc đăng ký trước tại `docs/ai/preregistration_v4.md` và Mục 5 `ke_hoach_v4_notebook.md`:
1. **So sánh ghép cặp `v3 ↔ v4` (`paired_tests_v3_v4.csv`):** Với mỗi kiến trúc (`DarkIR-Edge (ours)`, `Standard U-Net`), tính $\Delta\text{PSNR} = \text{PSNR}(\text{v4}) - \text{PSNR}(\text{v3})$ trên từng ảnh test (trung bình qua 2 seed `[42, 123]`), kiểm định Bootstrap 95% CI (2.000 lần, `seed = 42`), Paired t-test và Wilcoxon signed-rank test trên:
   - `lolv2+huawei` (`130` ảnh): **Nhóm bảo vệ khả năng tăng sáng** (điều kiện bắt buộc: `ci_lo >= -0.3 dB`).
   - `blur` (`40` ảnh): Nhóm kiểm tra điểm cộng khử mờ.
   - `nikon` (`20` ảnh): Ghi nhãn rõ `"v3: miền chưa thấy (zero-shot) / v4: miền đã thấy (in-domain)"` — không dùng làm phép thử đối chứng.
   - `pooled` (`190` ảnh) và từng miền thành phần (`lolv2`, `huawei`).
2. **Đánh giá không tham chiếu trên `Real-LOLBlur` (`482` ảnh RealBlur-J từ `real_lolblur_eval.zip`):**
   - Đo **NIQE, NRQM, MUSIQ** (khớp Bảng 5 bài báo DarkIR) qua `pyiqa`, kèm độ chói trung bình đầu ra/đầu vào và tỉ lệ pixel bão hòa ($\ge 250$).
   - Xuất `realblur_noref.csv`, `table_realblur_noref.csv` và lưới ảnh định tính `10` ảnh cố định theo `MASTER_SEED = 42` vào `realblur_visual_samples/`.


In [23]:
# ===== CELL 25 (MỚI - v4): Kiểm định ghép cặp v3 <-> v4 (paired_tests_v3_v4.csv) & Đánh giá Real-LOLBlur (482 ảnh) =====
# --- 5.1: So sánh ghép cặp v3 <-> v4 (paired_tests_v3_v4.csv) ---
v3_csv_candidates = [
    Path(CFG['v3_results_dir']) / 'results_per_image.csv',
    Path(DRIVE).parent / 'llie_project_v3' / 'experiments' / 'v3' / 'results' / 'results_per_image.csv',
    Path('colab-results/llie_project_v3-20261102/llie_project_v3/experiments/v3/results/results_per_image.csv'),
]
v3_csv_path = None if TEST_MODE else next((p for p in v3_csv_candidates if p.exists()), None)

if v3_csv_path is None and TEST_MODE:
    # Trong TEST_MODE, tạo bản giả lập v3 từ RES hiện tại (cộng nhiễu nhỏ xác định) để kiểm thử trọn vẹn pipeline paired_tests_v3_v4
    os.makedirs(CFG['v3_results_dir'], exist_ok=True)
    v3_csv_path = Path(CFG['v3_results_dir']) / 'results_per_image.csv'
    df_fake_v3 = RES.copy()
    rng_f = np.random.default_rng(MASTER_SEED)
    df_fake_v3['psnr'] = df_fake_v3['psnr'] - rng_f.normal(0.15, 0.1, size=len(df_fake_v3))
    df_fake_v3.to_csv(v3_csv_path, index=False)

if v3_csv_path is not None and v3_csv_path.exists():
    RES_V3 = pd.read_csv(v3_csv_path)
    print(f'✔ Đã nạp kết quả đối chứng v3 từ: {v3_csv_path} ({len(RES_V3)} dòng)')

    GROUP_DEFS = [
        ('lolv2+huawei', ['lolv2', 'huawei'], 'Nhóm bảo vệ tăng sáng (130 ảnh; điều kiện bắt buộc: ci_lo >= -0.3 dB)'),
        ('blur',         ['blur'],            'Điểm cộng khử mờ (40 ảnh; cần CI loại 0 và đạt điều kiện bảo vệ)'),
        ('nikon',        ['nikon'],           'v3: miền chưa thấy (zero-shot) / v4: miền đã thấy (in-domain) - không dùng làm đối chứng'),
        ('lolv2',        ['lolv2'],           'Miền thành phần tăng sáng LOLv2-Real (100 ảnh)'),
        ('huawei',       ['huawei'],          'Miền thành phần tăng sáng LSRW-Huawei (30 ảnh)'),
        ('pooled',       None,                'Gộp toàn bộ 190 ảnh (chỉ tham khảo do gồm cả nikon in-domain ở v4)'),
    ]

    def paired_v4_vs_v3(meth_name, dom_name, dom_list, note):
        s4 = RES[RES.method == meth_name]
        s3 = RES_V3[RES_V3.method == meth_name]
        if dom_list is not None:
            s4 = s4[s4.domain.isin(dom_list)]
            s3 = s3[s3.domain.isin(dom_list)]
        p4 = s4.groupby(['domain', 'image']).psnr.mean()
        p3 = s3.groupby(['domain', 'image']).psnr.mean()
        idx = p4.index.intersection(p3.index)
        d = (p4.loc[idx] - p3.loc[idx]).values
        if len(d) < 2:
            return None
        rng = np.random.default_rng(MASTER_SEED)
        ci = np.percentile([rng.choice(d, len(d)).mean() for _ in range(2000)], [2.5, 97.5])
        t_pval = stats.ttest_rel(p4.loc[idx], p3.loc[idx]).pvalue
        w_pval = stats.wilcoxon(d).pvalue if np.any(d != 0) else 1.0
        return dict(
            a=f'{meth_name} [v4]', b=f'{meth_name} [v3]',
            domain=dom_name, n=len(d),
            mean_v4=float(p4.loc[idx].mean()), mean_v3=float(p3.loc[idx].mean()),
            mean_delta_psnr=float(d.mean()), ci_lo=float(ci[0]), ci_hi=float(ci[1]),
            t_p=float(t_pval), wilcoxon_p=float(w_pval),
            frac_a_better=float((d > 0).mean()),
            group_note=note
        )

    v3_v4_rows = []
    for meth in ('DarkIR-Edge (ours)', 'Standard U-Net', 'DarkIR-Edge (no FFT)'):
        if meth in RES.method.values and meth in RES_V3.method.values:
            for g_name, g_doms, g_note in GROUP_DEFS:
                r = paired_v4_vs_v3(meth, g_name, g_doms, g_note)
                if r is not None:
                    v3_v4_rows.append(r)

    if v3_v4_rows:
        PAIRED_V3_V4 = pd.DataFrame(v3_v4_rows)
        p_v3_v4_path = f"{CFG['result_dir']}/paired_tests_v3_v4.csv"
        PAIRED_V3_V4.to_csv(p_v3_v4_path, index=False)
        if os.path.exists(CFG['latest_dir']):
            PAIRED_V3_V4.to_csv(f"{CFG['latest_dir']}/paired_tests_v3_v4.csv", index=False)
        print('=== KIỂM ĐỊNH ĐỐI CHỨNG ΔPSNR (v4 − v3) THEO QUY TẮC ĐĂNG KÝ TRƯỚC ===')
        cols_disp = ['a', 'domain', 'n', 'mean_v3', 'mean_v4', 'mean_delta_psnr', 'ci_lo', 'ci_hi', 't_p', 'wilcoxon_p', 'frac_a_better']
        print(PAIRED_V3_V4[cols_disp].round(4).to_string(index=False))
        print(f'✔ Đã lưu bảng so sánh v3 <-> v4 vào: {p_v3_v4_path}')
else:
    print(f'INFO: Chưa tìm thấy results_per_image.csv của v3 tại {CFG["v3_results_dir"]}. Đặt file kết quả v3 vào thư mục trên để tự động xuất paired_tests_v3_v4.csv.')

# --- 5.3: Đánh giá không tham chiếu trên Real-LOLBlur (482 ảnh từ real_lolblur_eval.zip) ---
realblur_root = Path(CFG['data_root']) / 'RealLOLBlur'
if not realblur_root.exists() and os.path.exists(CFG['realblur_zip']):
    print(f'Đang giải nén {CFG["realblur_zip"]} vào {realblur_root}...')
    realblur_root.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(CFG['realblur_zip']) as zf:
        zf.extractall(realblur_root)

if realblur_root.exists():
    rb_imgs = sorted([p for p in realblur_root.rglob('*') if p.is_file() and p.suffix.lower() in IMG_EXT])
    print(f'Tìm thấy {len(rb_imgs)} ảnh Real-LOLBlur tại {realblur_root}')
    if rb_imgs:
        PYIQA_OK = False
        if not TEST_MODE:
            try:
                import pyiqa
                PYIQA_OK = True
            except ImportError:
                if IN_COLAB:
                    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyiqa'])
                    try:
                        import pyiqa; PYIQA_OK = True
                    except Exception as e:
                        print('WARN: Không cài được pyiqa:', e)
        iqa_metrics = {}
        if PYIQA_OK:
            for m_name in ('niqe', 'nrqm', 'musiq'):
                try:
                    iqa_metrics[m_name] = pyiqa.create_metric(m_name, device=DEV)
                except Exception as e:
                    print(f'WARN: Không khởi tạo được chỉ số {m_name} trong pyiqa: {e}')

        @torch.no_grad()
        def noref_image_stats(out_u8, inp_u8):
            y_out = 0.299 * out_u8[..., 0].astype(np.float64) + 0.587 * out_u8[..., 1].astype(np.float64) + 0.114 * out_u8[..., 2].astype(np.float64)
            y_in  = 0.299 * inp_u8[..., 0].astype(np.float64)  + 0.587 * inp_u8[..., 1].astype(np.float64)  + 0.114 * inp_u8[..., 2].astype(np.float64)
            res = {
                'mean_lum_in': float(y_in.mean()),
                'mean_lum_out': float(y_out.mean()),
                'sat_ratio_out': float((out_u8 >= 250).any(axis=-1).mean()),
                'lap_var_out': float(cv2.Laplacian(y_out, cv2.CV_64F).var()),
            }
            if iqa_metrics:
                t_out = to_tensor(out_u8).unsqueeze(0).to(DEV)
                for m_name, metric_fn in iqa_metrics.items():
                    res[m_name] = float(metric_fn(t_out).item())
            return res

        rb_rows = []
        for tag, seeds in [('darkir_edge', CFG['darkir_edge_seeds']), ('unet', CFG['unet_seeds'])]:
            for s in seeds:
                ck = find_checkpoint(tag, s, ckpt_name='final.pt')
                if ck is None:
                    continue
                fn = nn_fn(tag, s, ckpt_name=ck.name)
                print(f'Đang đánh giá không tham chiếu Real-LOLBlur ({len(rb_imgs)} ảnh): {NAME_MAP[tag]} [v4, seed={s}]...')
                for p in rb_imgs:
                    inp = imread_rgb(str(p))
                    out = fn(inp)
                    rb_rows.append({'version': 'v4', 'method': NAME_MAP[tag], 'seed': s, 'image': p.name, **noref_image_stats(out, inp)})

        if rb_rows:
            DF_RB = pd.DataFrame(rb_rows)
            rb_csv = f"{CFG['result_dir']}/realblur_noref.csv"
            DF_RB.to_csv(rb_csv, index=False)
            if os.path.exists(CFG['latest_dir']):
                DF_RB.to_csv(f"{CFG['latest_dir']}/realblur_noref.csv", index=False)
            num_cols = [c for c in ('niqe', 'nrqm', 'musiq', 'mean_lum_in', 'mean_lum_out', 'sat_ratio_out', 'lap_var_out') if c in DF_RB.columns]
            print('=== BẢNG TÓM TẮT REAL-LOLBLUR (KHÔNG THAM CHIẾU) ===')
            print(DF_RB.groupby(['version', 'method'])[num_cols].agg(['mean', 'std']).round(3))

            # Xuất 10 ảnh định tính cố định theo MASTER_SEED = 42
            rng_rb = random.Random(MASTER_SEED)
            sample_rb = rng_rb.sample(rb_imgs, min(10, len(rb_imgs)))
            rb_vis_dir = Path(CFG['result_dir']) / 'realblur_visual_samples'
            rb_vis_dir.mkdir(parents=True, exist_ok=True)
            for idx_p, p in enumerate(sample_rb, 1):
                inp = imread_rgb(str(p))
                cv2.imwrite(str(rb_vis_dir / f'{idx_p:02d}_{p.stem}_input.png'), cv2.cvtColor(inp, cv2.COLOR_RGB2BGR))
                for tag in ('darkir_edge', 'unet'):
                    seeds = CFG['darkir_edge_seeds'] if tag == 'darkir_edge' else CFG['unet_seeds']
                    if seeds and find_checkpoint(tag, seeds[0]) is not None:
                        fn = nn_fn(tag, seeds[0])
                        cv2.imwrite(str(rb_vis_dir / f'{idx_p:02d}_{p.stem}_{tag}_v4.png'), cv2.cvtColor(fn(inp), cv2.COLOR_RGB2BGR))
            print(f'✔ Đã lưu kết quả Real-LOLBlur tại {rb_csv} và ảnh mẫu tại {rb_vis_dir}')
else:
    print(f'INFO: Chưa có gói {CFG["realblur_zip"]} trên Drive -> bỏ qua bước đánh giá phụ Real-LOLBlur.')


✔ Đã nạp kết quả đối chứng v3 từ: /content/drive/MyDrive/llie_project_v3/experiments/v3/results/results_per_image.csv (1900 dòng)
=== KIỂM ĐỊNH ĐỐI CHỨNG ΔPSNR (v4 − v3) THEO QUY TẮC ĐĂNG KÝ TRƯỚC ===
                        a       domain   n  mean_v3  mean_v4  mean_delta_psnr   ci_lo   ci_hi    t_p  wilcoxon_p  frac_a_better
  DarkIR-Edge (ours) [v4] lolv2+huawei 130  20.9775  20.7928          -0.1847 -0.5190  0.1005 0.2395      0.7231         0.5308
  DarkIR-Edge (ours) [v4]         blur  40  23.5684  23.5366          -0.0319 -0.5341  0.4330 0.8968      0.6463         0.5500
  DarkIR-Edge (ours) [v4]        nikon  20  16.9552  16.9611           0.0059 -0.6246  0.6570 0.9863      0.8695         0.5000
  DarkIR-Edge (ours) [v4]        lolv2 100  21.0715  20.7402          -0.3314 -0.7431  0.0274 0.0935      0.6401         0.5000
  DarkIR-Edge (ours) [v4]       huawei  30  20.6639  20.9682           0.3043  0.0041  0.6105 0.0613      0.0803         0.6333
  DarkIR-Edge (ours) [v4]      